Reference

https://www.kaggle.com/code/llkh0a/stanford-rna-3d-folding-part-2-protenix-tbm

# 日本語コメント付き版

この版では，元の処理はできるだけ変えずに，日本語コメントを追加しています．  
特に次の点が追いやすいようにしています．

- 各セルや関数の役割
- その処理が必要な理由
- TBM / Protenix / fallback のつながり
- 長鎖RNAを chunk に分けて再結合する意図


In [ ]:
import sys
print(sys.executable)

In [1]:
# Kaggle でもローカルでも同じ wheel を使えるように，入力ルートを自動判定します．
# Kaggle 側のパスを最優先し，見つからない場合だけ ./kaggle/input を使います．

from pathlib import Path
import subprocess
import sys

input_root = Path('/kaggle/input')
if not input_root.exists():
    input_root = Path.cwd() / 'kaggle' / 'input'

wheel_paths = [
    input_root / 'datasets' / 'kami1976' / 'biopython-cp312' / 'biopython-1.86-cp312-cp312-manylinux2014_x86_64.manylinux_2_17_x86_64.manylinux_2_28_x86_64.whl',
    input_root / 'datasets' / 'amirrezaaleyasin' / 'biotite' / 'biotite-1.6.0-cp312-cp312-manylinux_2_24_x86_64.manylinux_2_28_x86_64.whl',
    input_root / 'datasets' / 'amirrezaaleyasin' / 'rdkit-2025-9-5' / 'rdkit-2025.9.5-cp312-cp312-manylinux_2_28_x86_64.whl',
]

for wheel in wheel_paths:
    if wheel.exists():
        print(f'Installing: {wheel}')
        subprocess.run([
            sys.executable, '-m', 'pip', 'install', '--no-index', '--no-deps', str(wheel)
        ], check=True)
    else:
        print(f'Skipped (not found): {wheel}')


Processing /kaggle/input/datasets/kami1976/biopython-cp312/biopython-1.86-cp312-cp312-manylinux2014_x86_64.manylinux_2_17_x86_64.manylinux_2_28_x86_64.whl
biopython is already installed with the same version as the provided wheel. Use --force-reinstall to force an installation of the wheel.
Processing /kaggle/input/datasets/amirrezaaleyasin/biotite/biotite-1.6.0-cp312-cp312-manylinux_2_24_x86_64.manylinux_2_28_x86_64.whl
biotite is already installed with the same version as the provided wheel. Use --force-reinstall to force an installation of the wheel.
Processing /kaggle/input/datasets/amirrezaaleyasin/rdkit-2025-9-5/rdkit-2025.9.5-cp312-cp312-manylinux_2_28_x86_64.whl
  Attempting uninstall: rdkit
    Found existing installation: rdkit 2025.9.6
    Uninstalling rdkit-2025.9.6:
      Successfully uninstalled rdkit-2025.9.6


In [2]:
# 実行モードの切り替えと，テスト件数の制御を行うセルです．
# Kaggle本番では全件を処理し，ローカル検証では件数を絞って高速に確認できるようにします．

import os
import sys
import pandas as pd

# ── Local vs Kaggle mode ─────────────────────────────────────────────────────
# On Kaggle competition rerun, KAGGLE_IS_COMPETITION_RERUN is set to a truthy value.
# We also treat /kaggle/input existence as Kaggle runtime.

IS_KAGGLE = bool(os.environ.get('KAGGLE_IS_COMPETITION_RERUN', '')) or os.path.isdir('/kaggle/input')

# How many test samples to use when running locally
LOCAL_N_SAMPLES = None

if IS_KAGGLE:
    print('Running in KAGGLE COMPETITION mode — all test targets will be processed.')
else:
    print(f'Running in LOCAL mode — only the first {LOCAL_N_SAMPLES} test targets '
          f'will be processed to save time.')


Running in KAGGLE COMPETITION mode — all test targets will be processed.


In [3]:
# 以降で使う主要ライブラリを読み込みます．
# あわせて，Protenix 側の挙動に影響する環境変数もここで設定します．

import gc
import gzip
import json
import os
import time

os.environ["LAYERNORM_TYPE"] = "torch"
os.environ.setdefault("RNA_MSA_DEPTH_LIMIT", "512")

import sys
from pathlib import Path

import numpy as np
import pandas as pd
import torch
from Bio.Align import PairwiseAligner
from tqdm import tqdm

In [4]:
# Protenix や template 構造から，提出形式に必要な C1' 原子の位置を抜き出すための補助関数です．
# データ表現が複数あり得るため，属性・feature dict・ヒューリスティックの順に安全側で判定します．

def get_c1_mask(data: dict, atom_array) -> torch.Tensor:
    # 1. Try atom_array attributes first
    if atom_array is not None:
        try:
            if hasattr(atom_array, "centre_atom_mask"):
                m = atom_array.centre_atom_mask == 1
                if hasattr(atom_array, "is_rna"):
                    m = m & atom_array.is_rna
                return torch.from_numpy(m).bool()
            
            if hasattr(atom_array, "atom_name"):
                base = atom_array.atom_name == "C1'"
                if hasattr(atom_array, "is_rna"):
                    base = base & atom_array.is_rna
                return torch.from_numpy(base).bool()
        except Exception:
            pass

    # 2. Fallback to feature dict
    f = data["input_feature_dict"]
    
    if "centre_atom_mask" in f:
        return (f["centre_atom_mask"] == 1).bool()
    if "center_atom_mask" in f:
        return (f["center_atom_mask"] == 1).bool()
        
    # Heuristic fallback: check which index gives us roughly N_token atoms
    n_tokens = data.get("N_token", torch.tensor(0)).item()
    mask11 = (f["atom_to_tokatom_idx"] == 11).bool()
    mask12 = (f["atom_to_tokatom_idx"] == 12).bool()
    
    c11 = mask11.sum().item()
    c12 = mask12.sum().item()
    
    # Return the one closer to N_tokens (likely one per residue)
    if abs(c11 - n_tokens) < abs(c12 - n_tokens):
        return mask11
    else:
        return mask12

In [5]:

# ノートブック全体で共有するパス，モデル名，サンプル数，しきい値などを定義します．
# 実験条件を一か所に集めることで，設定変更時の事故を減らします．

KAGGLE_INPUT_ROOT = Path('/kaggle/input')
HOME_INPUT_ROOT = Path.home() / 'kaggle' / 'input'
LOCAL_INPUT_ROOT = Path.cwd() / 'kaggle' / 'input'

if KAGGLE_INPUT_ROOT.exists():
    DATA_INPUT_ROOT = KAGGLE_INPUT_ROOT
elif (HOME_INPUT_ROOT / 'stanford-rna-3d-folding-2').exists():
    DATA_INPUT_ROOT = HOME_INPUT_ROOT
else:
    DATA_INPUT_ROOT = LOCAL_INPUT_ROOT

if KAGGLE_INPUT_ROOT.exists():
    ASSET_INPUT_ROOT = KAGGLE_INPUT_ROOT
elif (LOCAL_INPUT_ROOT / 'datasets').exists():
    ASSET_INPUT_ROOT = LOCAL_INPUT_ROOT
else:
    ASSET_INPUT_ROOT = HOME_INPUT_ROOT

WORKING_ROOT = Path('/kaggle/working') if Path('/kaggle/working').exists() else (Path.cwd() / 'kaggle' / 'working')

DATA_BASE              = os.environ.get('DATA_BASE', str(DATA_INPUT_ROOT / 'stanford-rna-3d-folding-2'))
DEFAULT_TEST_CSV       = f"{DATA_BASE}/test_sequences.csv"
DEFAULT_TRAIN_CSV      = f"{DATA_BASE}/train_sequences.csv"
DEFAULT_TRAIN_LBLS     = f"{DATA_BASE}/train_labels.csv"
DEFAULT_VAL_CSV        = f"{DATA_BASE}/validation_sequences.csv"
DEFAULT_VAL_LBLS       = f"{DATA_BASE}/validation_labels.csv"
DEFAULT_OUTPUT         = str(WORKING_ROOT / 'submission.csv')

DEFAULT_CODE_DIR = os.environ.get(
    'PROTENIX_CODE_DIR_DEFAULT',
    str(
        ASSET_INPUT_ROOT
        / 'datasets'
        / 'qiweiyin'
        / 'protenix-v1-adjusted'
        / 'Protenix-v1-adjust-v2'
        / 'Protenix-v1-adjust-v2'
        / 'Protenix-v1'
    ),
)
DEFAULT_ROOT_DIR = DEFAULT_CODE_DIR

MODEL_NAME    = "protenix_base_20250630_v1.0.0"
N_SAMPLE      = 10
SEED          = 42
MAX_SEQ_LEN   = int(os.environ.get("MAX_SEQ_LEN",   "512"))
CHUNK_OVERLAP = int(os.environ.get("CHUNK_OVERLAP",  "128"))

# TBM quality thresholds — sequences below these get routed to Protenix
MIN_SIMILARITY = float(os.environ.get("MIN_SIMILARITY", "0.0"))

# Dynamic identity threshold: short RNAs are looser, long/high-gap alignments are stricter
REQ_PCT_ID_BASE            = float(os.environ.get("REQ_PCT_ID_BASE",            "34.0"))
REQ_PCT_ID_MIN             = float(os.environ.get("REQ_PCT_ID_MIN",             "28.0"))
REQ_PCT_ID_MAX             = float(os.environ.get("REQ_PCT_ID_MAX",             "72.0"))
REQ_PCT_ID_LEN_REF         = float(os.environ.get("REQ_PCT_ID_LEN_REF",         "120.0"))
REQ_PCT_ID_LEN_SLOPE       = float(os.environ.get("REQ_PCT_ID_LEN_SLOPE",       "18.0"))
REQ_PCT_ID_GAP_REF         = float(os.environ.get("REQ_PCT_ID_GAP_REF",         "0.08"))
REQ_PCT_ID_GAP_SLOPE       = float(os.environ.get("REQ_PCT_ID_GAP_SLOPE",       "90.0"))
REQ_PCT_ID_SHORT_BONUS_LEN = float(os.environ.get("REQ_PCT_ID_SHORT_BONUS_LEN", "70.0"))
REQ_PCT_ID_SHORT_BONUS     = float(os.environ.get("REQ_PCT_ID_SHORT_BONUS",     "6.0"))
LEGACY_MIN_PERCENT_IDENTITY = float(os.environ.get("LEGACY_MIN_PERCENT_IDENTITY", "50.0"))

# Rule-based TBM budget routing constants
TBM_DYNAMIC_MIN_TOP1_SIM          = float(os.environ.get("TBM_DYNAMIC_MIN_TOP1_SIM",          "0.45"))
TBM_DYNAMIC_LEVEL2_SIM            = float(os.environ.get("TBM_DYNAMIC_LEVEL2_SIM",            "0.62"))
TBM_DYNAMIC_LEVEL2_PCT            = float(os.environ.get("TBM_DYNAMIC_LEVEL2_PCT",            "45.0"))
TBM_DYNAMIC_LEVEL3_SIM            = float(os.environ.get("TBM_DYNAMIC_LEVEL3_SIM",            "0.74"))
TBM_DYNAMIC_LEVEL3_PCT            = float(os.environ.get("TBM_DYNAMIC_LEVEL3_PCT",            "55.0"))
TBM_DYNAMIC_LEVEL4_SIM            = float(os.environ.get("TBM_DYNAMIC_LEVEL4_SIM",            "0.82"))
TBM_DYNAMIC_LEVEL4_PCT            = float(os.environ.get("TBM_DYNAMIC_LEVEL4_PCT",            "65.0"))
TBM_DYNAMIC_LEVEL5_SIM            = float(os.environ.get("TBM_DYNAMIC_LEVEL5_SIM",            "0.90"))
TBM_DYNAMIC_LEVEL5_PCT            = float(os.environ.get("TBM_DYNAMIC_LEVEL5_PCT",            "78.0"))
TBM_DYNAMIC_GOOD_GAP              = float(os.environ.get("TBM_DYNAMIC_GOOD_GAP",              "0.12"))
TBM_DYNAMIC_STRICT_GAP            = float(os.environ.get("TBM_DYNAMIC_STRICT_GAP",            "0.06"))
TBM_DYNAMIC_HIGH_GAP_THRESHOLD    = float(os.environ.get("TBM_DYNAMIC_HIGH_GAP_THRESHOLD",    "0.22"))
TBM_DYNAMIC_HIGH_GAP_MAX_BUDGET   = int(os.environ.get("TBM_DYNAMIC_HIGH_GAP_MAX_BUDGET",     "2"))
TBM_DYNAMIC_GOOD_LEN_RATIO_DEV    = float(os.environ.get("TBM_DYNAMIC_GOOD_LEN_RATIO_DEV",    "0.18"))
TBM_DYNAMIC_STRICT_LEN_RATIO_DEV  = float(os.environ.get("TBM_DYNAMIC_STRICT_LEN_RATIO_DEV",  "0.08"))
TBM_DYNAMIC_BAD_LEN_RATIO_DEV     = float(os.environ.get("TBM_DYNAMIC_BAD_LEN_RATIO_DEV",     "0.28"))
TBM_DYNAMIC_BAD_LEN_RATIO_MAX_BUDGET = int(os.environ.get("TBM_DYNAMIC_BAD_LEN_RATIO_MAX_BUDGET", "2"))
TBM_DYNAMIC_MIN_TOP12_GAP         = float(os.environ.get("TBM_DYNAMIC_MIN_TOP12_GAP",         "0.03"))
TBM_DYNAMIC_LOW_TOP12_GAP         = float(os.environ.get("TBM_DYNAMIC_LOW_TOP12_GAP",         "0.015"))
TBM_DYNAMIC_LOW_TOP12_GAP_MAX_BUDGET = int(os.environ.get("TBM_DYNAMIC_LOW_TOP12_GAP_MAX_BUDGET", "3"))

# Multi-retrieval constants: 公開上位解法の複数検索器併用を軽量に模倣します
MULTI_RETRIEVAL_STANDARD_LENGTH_BAND = float(os.environ.get("MULTI_RETRIEVAL_STANDARD_LENGTH_BAND", "0.30"))
MULTI_RETRIEVAL_RELAXED_LENGTH_BAND  = float(os.environ.get("MULTI_RETRIEVAL_RELAXED_LENGTH_BAND",  "0.45"))
MULTI_RETRIEVAL_WEIGHT_GLOBAL        = float(os.environ.get("MULTI_RETRIEVAL_WEIGHT_GLOBAL",        "0.45"))
MULTI_RETRIEVAL_WEIGHT_LOCAL         = float(os.environ.get("MULTI_RETRIEVAL_WEIGHT_LOCAL",         "0.30"))
MULTI_RETRIEVAL_WEIGHT_PCT_ID        = float(os.environ.get("MULTI_RETRIEVAL_WEIGHT_PCT_ID",        "0.20"))
MULTI_RETRIEVAL_WEIGHT_GAP           = float(os.environ.get("MULTI_RETRIEVAL_WEIGHT_GAP",           "0.10"))
MULTI_RETRIEVAL_WEIGHT_LENGTH        = float(os.environ.get("MULTI_RETRIEVAL_WEIGHT_LENGTH",        "0.15"))

# Hybrid TBM-lite: 局所 alignment で良い断片だけを別 template から差し込みます
HYBRID_TBM_MIN_SEG_LEN             = int(os.environ.get("HYBRID_TBM_MIN_SEG_LEN", "10"))
HYBRID_TBM_MIN_LOCAL_SCORE         = float(os.environ.get("HYBRID_TBM_MIN_LOCAL_SCORE", "0.55"))
HYBRID_TBM_MIN_SEG_PCT_ID          = float(os.environ.get("HYBRID_TBM_MIN_SEG_PCT_ID", "55.0"))
HYBRID_TBM_MAX_DONORS              = int(os.environ.get("HYBRID_TBM_MAX_DONORS", "4"))
HYBRID_TBM_MAX_FRAGMENTS           = int(os.environ.get("HYBRID_TBM_MAX_FRAGMENTS", "2"))
HYBRID_TBM_ANCHOR_PAD              = int(os.environ.get("HYBRID_TBM_ANCHOR_PAD", "4"))
HYBRID_TBM_BOUNDARY_BLEND          = int(os.environ.get("HYBRID_TBM_BOUNDARY_BLEND", "3"))
HYBRID_TBM_MAX_SEGMENT_COVERAGE    = float(os.environ.get("HYBRID_TBM_MAX_SEGMENT_COVERAGE", "0.75"))

# Quality-aware stitching: overlap 品質に応じて chunk の重みを変えます
BLEND_MODE                         = os.environ.get("BLEND_MODE", "quality_aware").strip().lower()
STITCH_QUALITY_MIN_WEIGHT          = float(os.environ.get("STITCH_QUALITY_MIN_WEIGHT", "0.25"))
STITCH_QUALITY_RMSD_WEIGHT         = float(os.environ.get("STITCH_QUALITY_RMSD_WEIGHT", "0.85"))
STITCH_QUALITY_ADJ_WEIGHT          = float(os.environ.get("STITCH_QUALITY_ADJ_WEIGHT", "0.45"))
STITCH_QUALITY_CLASH_WEIGHT        = float(os.environ.get("STITCH_QUALITY_CLASH_WEIGHT", "0.25"))

# Cheap physics rerank: 候補群から極端に不自然な構造を下げます
PHYSICS_ADJ_TARGET                 = float(os.environ.get("PHYSICS_ADJ_TARGET", "5.95"))
PHYSICS_CLASH_DISTANCE             = float(os.environ.get("PHYSICS_CLASH_DISTANCE", "3.0"))
PHYSICS_MAX_POINTS                 = int(os.environ.get("PHYSICS_MAX_POINTS", "160"))
PHYSICS_WEIGHT_ADJ                 = float(os.environ.get("PHYSICS_WEIGHT_ADJ", "1.00"))
PHYSICS_WEIGHT_CLASH               = float(os.environ.get("PHYSICS_WEIGHT_CLASH", "1.20"))
PHYSICS_WEIGHT_COMPACT             = float(os.environ.get("PHYSICS_WEIGHT_COMPACT", "0.60"))

# Diversity-aware template selection: 似た template ばかり採らないようにします
DIVERSITY_MMR_ALPHA                = float(os.environ.get("DIVERSITY_MMR_ALPHA", "1.00"))
DIVERSITY_MMR_BETA                 = float(os.environ.get("DIVERSITY_MMR_BETA", "0.35"))

# Coordinate validation: Protenix や最終候補の壊れた座標を除外します
COORD_MAX_ADJ_DISTANCE             = float(os.environ.get("COORD_MAX_ADJ_DISTANCE", "12.0"))
COORD_MAX_BOX_SIZE                 = float(os.environ.get("COORD_MAX_BOX_SIZE", "1500.0"))
COORD_MAX_DUPLICATE_RATIO          = float(os.environ.get("COORD_MAX_DUPLICATE_RATIO", "0.20"))
COORD_MAX_SELF_CLASH_RATIO         = float(os.environ.get("COORD_MAX_SELF_CLASH_RATIO", "0.10"))

# Boltz-2 auxiliary branch: weak target にだけ補助候補を混ぜます
BOLTZ2_WEAK_TOP1_SIM               = float(os.environ.get("BOLTZ2_WEAK_TOP1_SIM", "0.58"))
BOLTZ2_WEAK_TOP1_PCT               = float(os.environ.get("BOLTZ2_WEAK_TOP1_PCT", "42.0"))
BOLTZ2_MAX_PRED_PER_TARGET         = int(os.environ.get("BOLTZ2_MAX_PRED_PER_TARGET", "2"))
BOLTZ2_CODE_DIR                    = os.environ.get("BOLTZ2_CODE_DIR", str(ASSET_INPUT_ROOT / "boltz2"))
BOLTZ2_MODEL_DIR                   = os.environ.get("BOLTZ2_MODEL_DIR", str(ASSET_INPUT_ROOT / "boltz2-models"))
BOLTZ2_CMD                         = os.environ.get("BOLTZ2_CMD", "").strip()

# Slot-level ensemble: source metadata を保持して並べ替えます
SLOT_ENSEMBLE_PRIMARY_SLOTS        = int(os.environ.get("SLOT_ENSEMBLE_PRIMARY_SLOTS", "6"))

# Set False to skip Protenix and use de-novo fallback instead
USE_PROTENIX = True

def parse_bool(value: str, default: bool = False) -> str:
    v = str(value).strip().lower()
    if v in {"1", "true", "t", "yes", "y", "on"}:
        return "true"
    if v in {"0", "false", "f", "no", "n", "off"}:
        return "false"
    return "true" if default else "false"


def env_flag(name: str, default: bool = False) -> bool:
    return parse_bool(os.environ.get(name, "true" if default else "false"), default) == "true"


USE_ADAPTIVE_IDENTITY_THRESHOLD = env_flag("USE_ADAPTIVE_IDENTITY_THRESHOLD", True)
USE_DYNAMIC_ROUTING = env_flag("USE_DYNAMIC_ROUTING", True)
USE_MULTI_RETRIEVAL = env_flag("USE_MULTI_RETRIEVAL", True)
USE_HYBRID_TBM_LITE = env_flag("USE_HYBRID_TBM_LITE", True)
USE_DIVERSITY_TEMPLATE_SELECTION = env_flag("USE_DIVERSITY_TEMPLATE_SELECTION", True)
USE_PHYSICS_RERANK = env_flag("USE_PHYSICS_RERANK", True)
USE_COORD_VALIDATION = env_flag("USE_COORD_VALIDATION", True)
USE_BOLTZ2_AUX = env_flag("USE_BOLTZ2_AUX", True)
USE_SLOT_ENSEMBLE = env_flag("USE_SLOT_ENSEMBLE", True)
USE_EXTERNAL_TEMPLATES = env_flag("USE_EXTERNAL_TEMPLATES", True)

# 既存のフラグ名も残して後方互換を保ちます
ENABLE_DYNAMIC_PCT_IDENTITY = USE_ADAPTIVE_IDENTITY_THRESHOLD
ENABLE_DYNAMIC_TBM_ALLOCATION = USE_DYNAMIC_ROUTING


USE_MSA      = parse_bool(os.environ.get("USE_MSA",      "false"))
USE_TEMPLATE = parse_bool(os.environ.get("USE_TEMPLATE", "false"))
USE_RNA_MSA  = parse_bool(os.environ.get("USE_RNA_MSA",  "true"))

# External template library: /kaggle/input に事前配置された JSON 群だけを追加投入します
EXTERNAL_TEMPLATE_ROOTS = [
    LOCAL_INPUT_ROOT / 'datasets' / 'fortbm_clustered_sampled_20250526_gt_re',
    LOCAL_INPUT_ROOT / 'datasets' / 'fortbm_clustered_sampled_20250526_gt_0521',
    LOCAL_INPUT_ROOT / 'datasets' / 'rnatbm-templates-20250521-re' / 'fortbm_clustered_sampled_20250526_gt_re',
    LOCAL_INPUT_ROOT / 'datasets' / 'rnatbm2025-set' / 'rna_tbm2025_clustered' / 'fortbm_clustered_sampled_20250526_gt_0521',
    ASSET_INPUT_ROOT / 'datasets' / 'fortbm_clustered_sampled_20250526_gt_re',
    ASSET_INPUT_ROOT / 'datasets' / 'fortbm_clustered_sampled_20250526_gt_0521',
    ASSET_INPUT_ROOT / 'datasets' / 'rnatbm-templates-20250521-re' / 'fortbm_clustered_sampled_20250526_gt_re',
    ASSET_INPUT_ROOT / 'datasets' / 'rnatbm2025-set' / 'rna_tbm2025_clustered' / 'fortbm_clustered_sampled_20250526_gt_0521',
    ASSET_INPUT_ROOT / 'datasets' / 'odat1248' / 'rnatbm-templates-20250521-re' / 'fortbm_clustered_sampled_20250526_gt_re',
    ASSET_INPUT_ROOT / 'datasets' / 'odat1248' / 'rnatbm2025-set' / 'rna_tbm2025_clustered' / 'fortbm_clustered_sampled_20250526_gt_0521',
]
EXTERNAL_TEMPLATE_GLOBS = ["*.json", "*.json.gz", "**/*.json", "**/*.json.gz"]
MIN_TEMPLATE_LEN = 3
MIN_RNA_FRACTION = 0.7
MIN_VALID_COORD_FRAC = 0.3
MIN_VALID_COORD_ABS = 3

MODEL_N_SAMPLE = int(os.environ.get("MODEL_N_SAMPLE", str(N_SAMPLE)))


# ─────────────── General Utilities ───────────────────────────────────────────
# 乱数の種を固定し，Kaggle上とローカル上で再現しやすい挙動に寄せます．
# cuDNN や CUDA も含めて決定論的な設定を有効化します．
def seed_everything(seed: int) -> None:
    os.environ["CUBLAS_WORKSPACE_CONFIG"] = ":4096:8"
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    np.random.seed(seed)
    torch.backends.cudnn.benchmark = False
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.enabled = True
    torch.use_deterministic_algorithms(True)


# 入力CSV，出力CSV，Protenixコードの配置場所を環境変数から解決します．
# 環境変数が未設定の場合は，Kaggle想定の既定値を使います．
def resolve_paths():
    test_csv   = os.environ.get("TEST_CSV",           DEFAULT_TEST_CSV)
    output_csv = os.environ.get("SUBMISSION_CSV",     DEFAULT_OUTPUT)
    code_dir   = os.environ.get("PROTENIX_CODE_DIR",  DEFAULT_CODE_DIR)
    root_dir   = os.environ.get("PROTENIX_ROOT_DIR",  DEFAULT_ROOT_DIR)
    return test_csv, output_csv, code_dir, root_dir


# Protenix の推論に必須な checkpoint と化学成分ファイルの存在を事前確認します．
# 後段で曖昧なエラーにするより，ここで明示的に止める方がデバッグしやすいです．
def ensure_required_files(root_dir: str) -> None:
    for p, name in [
        (Path(root_dir) / "checkpoint" / f"{MODEL_NAME}.pt",          "checkpoint"),
        (Path(root_dir) / "common" / "components.cif",                "CCD file"),
        (Path(root_dir) / "common" / "components.cif.rdkit_mol.pkl",  "CCD cache"),
    ]:
        if not p.exists():
            raise FileNotFoundError(f"Missing {name}: {p}")


# Protenix が受け取れる JSON 入力と config を組み立てる補助関数群です．
# notebook 内の DataFrame / 文字列設定を，モデル実行形式へ変換します．

# ─────────────── Protenix Input / Config Helpers ─────────────────────────────
# target_id と RNA 配列を，Protenix の入力 JSON 形式へ書き出します．
# 各ターゲットを name と sequence に変換し，後で InferenceDataset から読める形にします．
def build_input_json(df: pd.DataFrame, json_path: str) -> None:
    data = [
        {
            "name": row["target_id"],
            "covalent_bonds": [],
            "sequences": [{"rnaSequence": {"sequence": row["sequence"], "count": 1}}],
        }
        for _, row in df.iterrows()
    ]
    with open(json_path, "w", encoding="utf-8") as f:
        json.dump(data, f)


# Protenix の基本設定に対して，モデル種別や推論時の引数を上書きし，最終 config を作ります．
# notebook 側のフラグを，CLI 実行時と同じような引数文字列に落とし込みます．
def build_configs(input_json_path: str, dump_dir: str, model_name: str):
    from configs.configs_base import configs as configs_base
    from configs.configs_data import data_configs
    from configs.configs_inference import inference_configs
    from configs.configs_model_type import model_configs
    from protenix.config.config import parse_configs

    base = {**configs_base, **{"data": data_configs}, **inference_configs}

    def deep_update(t, p):
        for k, v in p.items():
            if isinstance(v, dict) and k in t and isinstance(t[k], dict):
                deep_update(t[k], v)
            else:
                t[k] = v

    deep_update(base, model_configs[model_name])
    arg_str = " ".join([
        f"--model_name {model_name}",
        f"--input_json_path {input_json_path}",
        f"--dump_dir {dump_dir}",
        f"--use_msa {USE_MSA}",
        f"--use_template {USE_TEMPLATE}",
        f"--use_rna_msa {USE_RNA_MSA}",
        f"--sample_diffusion.N_sample {MODEL_N_SAMPLE}",
        f"--seeds {SEED}",
    ])
    return parse_configs(configs=base, arg_str=arg_str, fill_required_with_null=True)


def get_c1_mask(data: dict, atom_array) -> torch.Tensor:
    # 1. Try atom_array attributes first
    if atom_array is not None:
        try:
            if hasattr(atom_array, "centre_atom_mask"):
                m = atom_array.centre_atom_mask == 1
                if hasattr(atom_array, "is_rna"):
                    m = m & atom_array.is_rna
                return torch.from_numpy(m).bool()
            
            if hasattr(atom_array, "atom_name"):
                base = atom_array.atom_name == "C1'"
                if hasattr(atom_array, "is_rna"):
                    base = base & atom_array.is_rna
                return torch.from_numpy(base).bool()
        except Exception:
            pass

    # 2. Fallback to feature dict
    f = data["input_feature_dict"]
    
    # CASE A: center_atom_mask exists
    if "center_atom_mask" in f:
        return (f["center_atom_mask"] == 1).bool()
    if "centre_atom_mask" in f:
        return (f["centre_atom_mask"] == 1).bool()
        
    # CASE B: Use atom_name
    if "atom_name" in f:
        # Check against "C1'" (byte encoded or string?)
        # For now assume typical behavior is center_atom_mask is present.
        pass

    # CASE C: atom_to_tokatom_idx fallback
    # The index for C1' is typically 11 or 12 depending on featurizer.
    # Let's try to match exactly C1' if possible.
    # But usually 'centre_atom_mask' should be there.
    
    # If we fall through, assume standard mask
    return (f["atom_to_tokatom_idx"] == 11).bool()


# feature dict だけから C1' 相当のマスクを作る簡易版です．
# 本体の get_c1_mask より単純ですが，feature のみ参照したい場面で使えます．
def get_feature_c1_mask(data: dict) -> torch.Tensor:
    f = data["input_feature_dict"]
    if "centre_atom_mask" in f:
        return f["centre_atom_mask"].long() == 1
    return f["atom_to_tokatom_idx"].long() == 12


# 座標テンソルを Kaggle 提出形式の1行ずつの辞書へ変換します．
# 1残基につき1行を作り，x_1〜z_10 までの列を埋めます．
def coords_to_rows(target_id: str, seq: str, coords: np.ndarray) -> list:
    """coords shape: (N_SAMPLE, seq_len, 3)"""
    rows = []
    for i in range(len(seq)):
        row = {"ID": f"{target_id}_{i + 1}", "resname": seq[i], "resid": i + 1}
        for s in range(N_SAMPLE):
            if s < coords.shape[0] and i < coords.shape[1]:
                x, y, z = coords[s, i]
            else:
                x, y, z = 0.0, 0.0, 0.0
            row[f"x_{s + 1}"] = float(x)
            row[f"y_{s + 1}"] = float(y)
            row[f"z_{s + 1}"] = float(z)
        rows.append(row)
    return rows


# 予測サンプル数が不足している場合に，先頭サンプルの複製で本数を揃えます．
# 提出形式では各 target ごとに固定本数が必要なため，保険として用意されています．
def pad_samples(coords: np.ndarray, n: int) -> np.ndarray:
    if coords.shape[0] >= n:
        return coords[:n]
    if coords.shape[0] == 0:
        return np.zeros((n, coords.shape[1], 3), dtype=coords.dtype)
    extra = np.repeat(coords[:1], n - coords.shape[0], axis=0)
    return np.concatenate([coords, extra], axis=0)


# 長いRNAを，Protenix が扱える最大長以下の重複チャンクへ分割します．
# overlap を持たせることで，後でチャンク同士を幾何的につなぎ直せるようにします．
def split_into_chunks(seq_len: int, max_len: int, overlap: int) -> list:
    """Split a sequence into overlapping (start, end) chunks."""
    if seq_len <= max_len:
        return [(0, seq_len)]
    chunks = []
    step = max_len - overlap
    pos = 0
    while pos < seq_len:
        end = min(pos + max_len, seq_len)
        chunks.append((pos, end))
        if end == seq_len:
            break
        pos += step
    return chunks


# 2つの点群 P, Q を最もよく一致させる剛体変換を求めます．
# チャンクの重複領域を整列させるための基本処理です．
def kabsch_align(P: np.ndarray, Q: np.ndarray):
    """Compute optimal rotation R and translation t so that  R @ P + t ≈ Q."""
    centroid_P = P.mean(axis=0)
    centroid_Q = Q.mean(axis=0)
    Pc = P - centroid_P
    Qc = Q - centroid_Q
    H = Pc.T @ Qc
    U, _, Vt = np.linalg.svd(H)
    d = np.linalg.det(Vt.T @ U.T)
    S = np.eye(3)
    if d < 0:
        S[2, 2] = -1
    R = Vt.T @ S @ U.T
    t = centroid_Q - R @ centroid_P
    return R, t


def _sample_long_range_points(coords: np.ndarray, max_points: int = PHYSICS_MAX_POINTS) -> np.ndarray:
    n = len(coords)
    if n <= max_points:
        return coords
    idx = np.linspace(0, n - 1, max_points).astype(int)
    return coords[idx]


def _self_clash_ratio(coords: np.ndarray, clash_dist: float = PHYSICS_CLASH_DISTANCE) -> float:
    P = _sample_long_range_points(coords)
    if len(P) < 4:
        return 0.0
    diff = P[:, None, :] - P[None, :, :]
    dm = np.linalg.norm(diff, axis=2) + 1e-6
    sep = np.abs(np.arange(len(P))[:, None] - np.arange(len(P))[None, :])
    mask = np.triu((sep > 2) & (dm < clash_dist), k=1)
    denom = max(mask.size, 1)
    return float(mask.sum() / denom)


def _adjacent_distance_error(coords: np.ndarray, target: float = PHYSICS_ADJ_TARGET) -> float:
    if len(coords) < 2:
        return 0.0
    d = np.linalg.norm(coords[1:] - coords[:-1], axis=1)
    return float(np.mean(np.abs(d - target)))


def _overlap_quality_metrics(prev_overlap: np.ndarray, cur_overlap: np.ndarray) -> dict:
    if len(prev_overlap) < 3 or len(cur_overlap) < 3:
        return {"rmsd": 9.9, "adj_error": 9.9, "clash_ratio": 1.0, "score": STITCH_QUALITY_MIN_WEIGHT}
    rmsd = float(np.sqrt(np.mean((prev_overlap - cur_overlap) ** 2)))
    adj_error = _adjacent_distance_error(cur_overlap)
    clash_ratio = _self_clash_ratio(cur_overlap)
    raw = np.exp(
        -STITCH_QUALITY_RMSD_WEIGHT * rmsd
        -STITCH_QUALITY_ADJ_WEIGHT * adj_error
        -STITCH_QUALITY_CLASH_WEIGHT * clash_ratio
    )
    score = float(np.clip(raw, STITCH_QUALITY_MIN_WEIGHT, 1.0))
    return {"rmsd": rmsd, "adj_error": adj_error, "clash_ratio": clash_ratio, "score": score}


# チャンクごとの予測座標を，全長RNAの1本の座標列へつなぎ直します．
# 重複部では Kabsch 整列を行い，quality-aware な重み付けブレンドも選べるようにします．
def stitch_chunk_coords(chunk_coords_list: list,
                        chunk_ranges: list,
                        seq_len: int) -> np.ndarray:
    """
    Merge overlapping chunk coordinates into a full sequence geometry.
    Applies Kabsch alignment on overlapping residues, and smoothly
    blends the coordinates using a linear weight ramp or a quality-aware ramp.
    """
    if len(chunk_coords_list) == 1:
        coords = chunk_coords_list[0]
        if coords.shape[0] >= seq_len:
            return coords[:seq_len]
        out = np.zeros((seq_len, 3), dtype=coords.dtype)
        out[:coords.shape[0]] = coords
        return out

    aligned = [chunk_coords_list[0].copy()]
    quality_scores = [1.0]

    for i in range(1, len(chunk_coords_list)):
        prev_start, prev_end = chunk_ranges[i - 1]
        cur_start, cur_end = chunk_ranges[i]
        ov_start = cur_start
        ov_end = min(prev_end, cur_end)
        ov_len = ov_end - ov_start

        if ov_len < 3:
            aligned.append(chunk_coords_list[i].copy())
            quality_scores.append(STITCH_QUALITY_MIN_WEIGHT)
            continue

        prev_ov = aligned[i - 1][ov_start - prev_start: ov_end - prev_start]
        cur_ov = chunk_coords_list[i][ov_start - cur_start: ov_end - cur_start]
        valid = ~(np.isnan(prev_ov).any(axis=1) | np.isnan(cur_ov).any(axis=1))
        if valid.sum() < 3:
            aligned.append(chunk_coords_list[i].copy())
            quality_scores.append(STITCH_QUALITY_MIN_WEIGHT)
            continue

        R, t = kabsch_align(cur_ov[valid], prev_ov[valid])
        transformed = (chunk_coords_list[i] @ R.T) + t
        cur_ov_aligned = transformed[ov_start - cur_start: ov_end - cur_start][valid]
        metrics = _overlap_quality_metrics(prev_ov[valid], cur_ov_aligned)
        aligned.append(transformed)
        quality_scores.append(metrics["score"])
        print(
            f"    stitch chunk{i}: rmsd={metrics['rmsd']:.3f}, "
            f"adj_err={metrics['adj_error']:.3f}, clash={metrics['clash_ratio']:.4f}, "
            f"weight={metrics['score']:.3f}"
        )

    full = np.zeros((seq_len, 3), dtype=np.float64)
    weights = np.zeros(seq_len, dtype=np.float64)

    for i, ((s, e), coords) in enumerate(zip(chunk_ranges, aligned)):
        chunk_len = coords.shape[0]
        actual_end = min(s + chunk_len, seq_len)
        used_len = actual_end - s
        w = np.ones(used_len, dtype=np.float64)

        if i > 0:
            ov_start = s
            ov_end = min(chunk_ranges[i - 1][1], e)
            ramp_len = ov_end - ov_start
            if ramp_len > 0:
                w[:ramp_len] = np.linspace(0.0, 1.0, ramp_len)

        if i < len(chunk_ranges) - 1:
            next_s = chunk_ranges[i + 1][0]
            ramp_start = next_s - s
            ramp_len = actual_end - next_s
            if ramp_len > 0 and ramp_start < used_len:
                w[ramp_start:used_len] = np.minimum(w[ramp_start:used_len], np.linspace(1.0, 0.0, ramp_len))

        if BLEND_MODE == "quality_aware":
            w = w * quality_scores[i]

        full[s:actual_end] += coords[:used_len] * w[:, None]
        weights[s:actual_end] += w

    mask = weights > 0
    full[mask] /= weights[mask, None]
    return full


# ここからは Template-Based Modeling の中核ロジックです．
# 既知構造の検索，query への写像，幾何補正，多様化生成を担います．

# ─────────────── TBM Core Functions ──────────────────────────────────────────
# 配列アラインメント器を初期化します．
# match / mismatch / gap penalty を RNA 向けの経験則に基づいて設定しています．
def _make_aligner(mode="global") -> PairwiseAligner:
    al = PairwiseAligner()
    al.mode                           = mode
    al.match_score                    = 2
    al.mismatch_score                 = -1.5
    al.open_gap_score                 = -8
    al.extend_gap_score               = -0.4
    al.query_left_open_gap_score      = -8
    al.query_left_extend_gap_score    = -0.4
    al.query_right_open_gap_score     = -8
    al.query_right_extend_gap_score   = -0.4
    al.target_left_open_gap_score     = -8
    al.target_left_extend_gap_score   = -0.4
    al.target_right_open_gap_score    = -8
    al.target_right_extend_gap_score  = -0.4
    return al


_aligner = _make_aligner("global")
_local_aligner = _make_aligner("local")


# 例えば 'A:2;B:1' のような化学量論表記を，(chain, 個数) のリストへ変換します．
# chain ごとの区間推定に使います．
def parse_stoichiometry(stoich: str) -> list:
    if pd.isna(stoich) or str(stoich).strip() == "":
        return []
    return [(ch.strip(), int(cnt)) for part in str(stoich).split(";")
            for ch, cnt in [part.split(":")]]


# FASTA文字列を chain_id -> 配列 の辞書へ変換します．
# all_sequences カラムの解析に使います．
def parse_fasta(fasta_content: str) -> dict:
    out, cur, parts = {}, None, []
    for line in str(fasta_content).splitlines():
        line = line.strip()
        if not line:
            continue
        if line.startswith(">"):
            if cur is not None:
                out[cur] = "".join(parts)
            cur = line[1:].split()[0]
            parts = []
        else:
            parts.append(line.replace(" ", ""))
    if cur is not None:
        out[cur] = "".join(parts)
    return out


# target の全配列を，chain ごとの開始位置・終了位置の区間へ分解します．
# stoichiometry と all_sequences が使えない場合は，全長を1本の区間として扱います．
def get_chain_segments(row) -> list:
    seq    = row["sequence"]
    stoich = row.get("stoichiometry", "")
    all_sq = row.get("all_sequences", "")
    if (pd.isna(stoich) or pd.isna(all_sq)
            or str(stoich).strip() == "" or str(all_sq).strip() == ""):
        return [(0, len(seq))]
    try:
        chain_dict = parse_fasta(all_sq)
        order = parse_stoichiometry(stoich)
        segs, pos = [], 0
        for ch, cnt in order:
            base = chain_dict.get(ch)
            if base is None:
                return [(0, len(seq))]
            for _ in range(cnt):
                segs.append((pos, pos + len(base)))
                pos += len(base)
        return segs if pos == len(seq) else [(0, len(seq))]
    except Exception:
        return [(0, len(seq))]


# 全 target について，chain 区間情報と stoichiometry を辞書化します．
# 後段の拘束補正や chain ごとの揺らぎ付与で使います．
def build_segments_map(df: pd.DataFrame) -> tuple:
    seg_map, stoich_map = {}, {}
    for _, r in df.iterrows():
        tid               = r["target_id"]
        seg_map[tid]      = get_chain_segments(r)
        raw_s             = r.get("stoichiometry", "")
        stoich_map[tid]   = "" if pd.isna(raw_s) else str(raw_s)
    return seg_map, stoich_map


# train / validation のラベルCSVから，target_id ごとの C1' 座標配列を取り出します．
# TBM で参照する template 構造ライブラリをここで構築します．
def process_labels(labels_df: pd.DataFrame) -> dict:
    coords = {}
    prefixes = labels_df["ID"].str.rsplit("_", n=1).str[0]
    for prefix, grp in labels_df.groupby(prefixes):
        coords[prefix] = grp.sort_values("resid")[["x_1", "y_1", "z_1"]].values
    return coords


def normalize_rna_sequence(seq) -> str:
    if seq is None:
        return ""
    seq = str(seq).upper().replace("T", "U")
    return "".join(ch for ch in seq if not ch.isspace())


def _safe_float3(x):
    if isinstance(x, (list, tuple)) and len(x) == 3:
        try:
            return [float(x[0]), float(x[1]), float(x[2])]
        except (TypeError, ValueError):
            pass
    return [np.nan, np.nan, np.nan]


def _extract_residue_name(residue) -> str:
    for key in ("one_letter_code", "resname", "name", "residue_name", "base", "nt"):
        value = residue.get(key)
        if not isinstance(value, str):
            continue
        token = value.strip().upper()
        if not token:
            continue
        if len(token) == 1:
            return "U" if token == "T" else token
        for ch in token:
            if ch in "ACGUT":
                return "U" if ch == "T" else ch
    return ""


def _extract_c1p_coords(residue):
    atoms = residue.get("atoms", {})
    if isinstance(atoms, dict):
        for key in ("C1'", "C1*", "c1p", "c1_prime"):
            if key in atoms:
                return _safe_float3(atoms[key])
    elif isinstance(atoms, list):
        for atom in atoms:
            if not isinstance(atom, dict):
                continue
            atom_name = str(atom.get("name") or atom.get("atom_name") or atom.get("atom") or "")
            if atom_name.replace("*", "'").upper() != "C1'":
                continue
            coord = atom.get("coord") or atom.get("coords") or atom.get("xyz")
            if coord is None:
                coord = [atom.get("x"), atom.get("y"), atom.get("z")]
            return _safe_float3(coord)
    return [np.nan, np.nan, np.nan]


def passes_template_quality_filter(sequence, coords):
    sequence = normalize_rna_sequence(sequence)
    if len(sequence) < MIN_TEMPLATE_LEN:
        return False, f"too_short<{MIN_TEMPLATE_LEN}"
    arr = np.asarray(coords, dtype=np.float64)
    if arr.ndim != 2 or arr.shape[1] != 3 or len(arr) != len(sequence):
        return False, f"bad_shape={arr.shape},len={len(sequence)}"
    valid_mask = np.isfinite(arr).all(axis=1)
    min_valid = max(MIN_VALID_COORD_ABS, int(np.ceil(MIN_VALID_COORD_FRAC * len(sequence))))
    if int(valid_mask.sum()) < min_valid:
        return False, f"valid_c1p<{min_valid}"
    rna_fraction = sum(ch in "ACGU" for ch in sequence) / len(sequence)
    if rna_fraction < MIN_RNA_FRACTION:
        return False, f"rna_fraction<{MIN_RNA_FRACTION:.2f}"
    return True, "ok"


def load_external_template_json(json_file):
    try:
        path = Path(json_file)
        opener = gzip.open if path.suffix == ".gz" else open
        with opener(path, "rt", encoding="utf-8") as f:
            data = json.load(f)

        residues = None
        if isinstance(data, list):
            residues = data
        elif isinstance(data, dict):
            for value in data.values():
                if isinstance(value, list):
                    residues = value
                    break
        if not isinstance(residues, list) or not residues:
            return None, None, "no_residue_list"

        sequence = []
        coords = []
        for residue in residues:
            if not isinstance(residue, dict):
                continue
            base = _extract_residue_name(residue)
            if not base:
                continue
            sequence.append(base)
            coords.append(_extract_c1p_coords(residue))

        sequence = normalize_rna_sequence("".join(sequence))
        if not sequence:
            return None, None, "empty_sequence"
        coords = np.asarray(coords, dtype=np.float64)
        ok, reason = passes_template_quality_filter(sequence, coords)
        if not ok:
            return None, None, reason
        return sequence, coords, "ok"
    except Exception as exc:
        return None, None, f"error:{type(exc).__name__}"


def discover_external_template_files():
    if not USE_EXTERNAL_TEMPLATES:
        print("External templates: disabled")
        return []

    valid_roots = []
    for root in EXTERNAL_TEMPLATE_ROOTS:
        root = Path(root)
        if root.exists():
            valid_roots.append(root)
            print(f"External templates: found root {root}")

    if not valid_roots:
        print("External templates: no roots found, skipping")
        return []

    discovered = {}
    for root in valid_roots:
        before = len(discovered)
        for pattern in EXTERNAL_TEMPLATE_GLOBS:
            for path in root.glob(pattern):
                if not path.is_file():
                    continue
                name = path.name
                if not (name.endswith('.json') or name.endswith('.json.gz')):
                    continue
                discovered[str(path)] = path
        print(f"External templates: {root} -> {len(discovered) - before} files")
    files = [discovered[key] for key in sorted(discovered)]
    print(f"External templates: discovered {len(files)} unique files")
    return files


def build_external_template_pool(existing_sequences):
    existing_sequences = {
        normalize_rna_sequence(seq)
        for seq in existing_sequences
        if normalize_rna_sequence(seq)
    }
    files = discover_external_template_files()
    if not files:
        return pd.DataFrame(columns=["target_id", "sequence"]), {}

    ext_rows = []
    ext_coords = {}
    stats = {
        "loaded": 0,
        "duplicate": 0,
        "filtered": 0,
    }
    filter_reasons = {}

    for path in files:
        sequence, coords, reason = load_external_template_json(path)
        if sequence is None:
            stats["filtered"] += 1
            filter_reasons[reason] = filter_reasons.get(reason, 0) + 1
            continue
        if sequence in existing_sequences:
            stats["duplicate"] += 1
            continue
        ext_id = f"EXT_{len(ext_rows) + 1:08d}"
        ext_rows.append({"target_id": ext_id, "sequence": sequence})
        ext_coords[ext_id] = coords
        existing_sequences.add(sequence)
        stats["loaded"] += 1

    print(
        "External templates: "
        f"loaded={stats['loaded']}, duplicate={stats['duplicate']}, filtered={stats['filtered']}"
    )
    if filter_reasons:
        summary = ", ".join(f"{k}={v}" for k, v in sorted(filter_reasons.items()))
        print(f"External templates: filtered reasons -> {summary}")
    return pd.DataFrame(ext_rows), ext_coords


def merge_external_templates_into_pool(all_train_seqs, all_train_coords, all_train_ids=None):
    ext_seqs, ext_coords = build_external_template_pool(all_train_seqs["sequence"].tolist())
    if ext_seqs.empty:
        print("External templates: no new templates merged")
        return all_train_seqs, all_train_coords, all_train_ids

    merged_seqs = pd.concat([all_train_seqs, ext_seqs], ignore_index=True)
    merged_coords = dict(all_train_coords)
    merged_coords.update(ext_coords)
    if all_train_ids is not None:
        new_ids = ext_seqs["target_id"].tolist()
        if hasattr(all_train_ids, "extend"):
            all_train_ids.extend(new_ids)
        elif hasattr(all_train_ids, "update"):
            all_train_ids.update(new_ids)
    print(
        f"Template pool after external merge: {len(merged_seqs)} sequences, "
        f"{len(merged_coords)} structures (+{len(ext_seqs)} external)"
    )
    return merged_seqs, merged_coords, all_train_ids


# alignment オブジェクトから，gap を含む query / template の整列文字列を復元します．
# 可視化やデバッグ，類似度の解釈をしやすくするための補助関数です．
def _build_aligned_strings(query_seq, template_seq, alignment):
    q_segs, t_segs = alignment.aligned
    aq, at, qi, ti = [], [], 0, 0
    for (qs, qe), (ts, te) in zip(q_segs, t_segs):
        while qi < qs: aq.append(query_seq[qi]);    at.append("-");              qi += 1
        while ti < ts: aq.append("-");              at.append(template_seq[ti]); ti += 1
        for qp, tp in zip(range(qs, qe), range(ts, te)):
            aq.append(query_seq[qp]); at.append(template_seq[tp])
        qi, ti = qe, te
    while qi < len(query_seq):    aq.append(query_seq[qi]);    at.append("-");              qi += 1
    while ti < len(template_seq): aq.append("-");              at.append(template_seq[ti]); ti += 1
    return "".join(aq), "".join(at)


# query 配列に近い既知配列を全候補から探索し，類似度順に上位候補を返します．
# 正規化アラインメントスコア，percent identity，整列済み文字列も併せて保持します．
def _count_identical_pairs(query_seq, template_seq, alignment):
    return sum(
        1 for (qs, qe), (ts, te) in zip(*alignment.aligned)
        for qp, tp in zip(range(qs, qe), range(ts, te))
        if query_seq[qp] == template_seq[tp]
    )


def _normalize_alignment_score(alignment, query_len, template_len):
    denom = max(2 * min(query_len, template_len), 1)
    return float(alignment.score) / denom


def _evaluate_alignment_bundle(query_seq, tid, tseq, tmpl_coords, global_aln=None, local_aln=None, retrieval_sources=None):
    retrieval_sources = sorted(set(retrieval_sources or []))
    primary_aln = global_aln if global_aln is not None else local_aln
    if primary_aln is None:
        return None
    aq, at = _build_aligned_strings(query_seq, tseq, primary_aln)
    aln_len = max(len(aq), 1)
    gap_fraction = sum(1 for q, t in zip(aq, at) if q == "-" or t == "-") / aln_len
    identical = _count_identical_pairs(query_seq, tseq, primary_aln)
    pct_id = 100 * identical / max(len(query_seq), 1)
    length_ratio = len(tseq) / max(len(query_seq), 1)
    global_score = _normalize_alignment_score(global_aln, len(query_seq), len(tseq)) if global_aln is not None else 0.0
    local_score = _normalize_alignment_score(local_aln, len(query_seq), len(tseq)) if local_aln is not None else 0.0
    rerank_score = (
        MULTI_RETRIEVAL_WEIGHT_GLOBAL * global_score
        + MULTI_RETRIEVAL_WEIGHT_LOCAL * local_score
        + MULTI_RETRIEVAL_WEIGHT_PCT_ID * (pct_id / 100.0)
        - MULTI_RETRIEVAL_WEIGHT_GAP * gap_fraction
        - MULTI_RETRIEVAL_WEIGHT_LENGTH * abs(length_ratio - 1.0)
    )
    return {
        "tid": tid,
        "tseq": tseq,
        "tmpl_coords": tmpl_coords,
        "global_score": global_score,
        "local_score": local_score,
        "sim": max(global_score, local_score),
        "pct_id": pct_id,
        "gap_fraction": gap_fraction,
        "length_ratio": length_ratio,
        "aq": aq,
        "at": at,
        "retrieval_sources": "+".join(retrieval_sources) if retrieval_sources else "global",
        "rerank_score": rerank_score,
    }


def _format_retrieval_summary(record):
    if record is None:
        return "none"
    return (
        f"src={record[9]}, sim={record[2]:.3f}, pct_id={record[4]:.1f}, "
        f"gap={record[5]:.3f}, len_ratio={record[6]:.3f}"
    )


def find_similar_sequences_detailed(query_seq, train_seqs_df, train_coords_dict, top_n=30):
    if not USE_MULTI_RETRIEVAL:
        results = []
        for _, row in train_seqs_df.iterrows():
            tid, tseq = row["target_id"], row["sequence"]
            if tid not in train_coords_dict:
                continue
            if abs(len(tseq) - len(query_seq)) / max(len(tseq), len(query_seq)) > MULTI_RETRIEVAL_STANDARD_LENGTH_BAND:
                continue
            aln = next(iter(_aligner.align(query_seq, tseq)))
            record = _evaluate_alignment_bundle(
                query_seq, tid, tseq, train_coords_dict[tid], global_aln=aln, retrieval_sources=["global"]
            )
            results.append((
                record["tid"], record["tseq"], record["sim"], record["tmpl_coords"], record["pct_id"],
                record["gap_fraction"], record["length_ratio"], record["aq"], record["at"], record["retrieval_sources"]
            ))
        results.sort(key=lambda x: x[2], reverse=True)
        return results[:top_n]

    merged = {}
    for _, row in train_seqs_df.iterrows():
        tid, tseq = row["target_id"], row["sequence"]
        if tid not in train_coords_dict:
            continue
        length_dev = abs(len(tseq) - len(query_seq)) / max(len(tseq), len(query_seq))
        allow_global = length_dev <= MULTI_RETRIEVAL_STANDARD_LENGTH_BAND
        allow_relaxed = length_dev <= MULTI_RETRIEVAL_RELAXED_LENGTH_BAND
        if not allow_relaxed:
            continue

        retrieval_sources = []
        global_aln = None
        if allow_global:
            global_aln = next(iter(_aligner.align(query_seq, tseq)))
            retrieval_sources.append("global")
        elif allow_relaxed:
            global_aln = next(iter(_aligner.align(query_seq, tseq)))
            retrieval_sources.append("relaxed")

        local_aln = next(iter(_local_aligner.align(query_seq, tseq)))
        if local_aln is not None and local_aln.score > 0:
            retrieval_sources.append("local")

        record = _evaluate_alignment_bundle(
            query_seq, tid, tseq, train_coords_dict[tid], global_aln=global_aln, local_aln=local_aln,
            retrieval_sources=retrieval_sources
        )
        if record is None:
            continue
        prev = merged.get(tid)
        if prev is None or record["rerank_score"] > prev["rerank_score"]:
            merged[tid] = record

    ranked = sorted(merged.values(), key=lambda r: (r["rerank_score"], r["sim"]), reverse=True)
    return [
        (
            r["tid"], r["tseq"], r["sim"], r["tmpl_coords"], r["pct_id"], r["gap_fraction"], r["length_ratio"],
            r["aq"], r["at"], r["retrieval_sources"]
        )
        for r in ranked[:top_n]
    ]


def required_pct_identity(seq_len, gap_fraction):
    if not ENABLE_DYNAMIC_PCT_IDENTITY:
        return LEGACY_MIN_PERCENT_IDENTITY
    seq_len = max(float(seq_len), 1.0)
    gap_fraction = min(max(float(gap_fraction), 0.0), 1.0)
    len_term = REQ_PCT_ID_LEN_SLOPE * (seq_len - REQ_PCT_ID_LEN_REF) / REQ_PCT_ID_LEN_REF
    gap_term = REQ_PCT_ID_GAP_SLOPE * (gap_fraction - REQ_PCT_ID_GAP_REF)
    short_bonus = REQ_PCT_ID_SHORT_BONUS if seq_len <= REQ_PCT_ID_SHORT_BONUS_LEN else 0.0
    required = REQ_PCT_ID_BASE + len_term + gap_term - short_bonus
    return float(np.clip(required, REQ_PCT_ID_MIN, REQ_PCT_ID_MAX))


def summarize_tbm_routing_features(seq_len, similar):
    if not similar:
        return {
            "top1_sim": 0.0,
            "top2_sim": 0.0,
            "top1_pct_id": 0.0,
            "top1_top2_gap": 0.0,
            "length_ratio": 1.0,
            "alignment_gap_fraction": 1.0,
        }
    top1 = similar[0]
    top2 = similar[1] if len(similar) > 1 else None
    top1_sim = float(top1[2])
    top2_sim = float(top2[2]) if top2 is not None else 0.0
    return {
        "top1_sim": top1_sim,
        "top2_sim": top2_sim,
        "top1_pct_id": float(top1[4]),
        "top1_top2_gap": top1_sim - top2_sim,
        "length_ratio": float(top1[6]),
        "alignment_gap_fraction": float(top1[5]),
    }


def decide_tbm_vs_protenix_allocation(seq_len, similar, features=None):
    feat = summarize_tbm_routing_features(seq_len, similar) if features is None else features
    if not similar:
        return 0, N_SAMPLE, feat, "no_template_candidates"
    if not USE_DYNAMIC_ROUTING:
        return N_SAMPLE, 0, feat, "dynamic_routing=off"
    if feat["top1_sim"] < TBM_DYNAMIC_MIN_TOP1_SIM:
        return 0, N_SAMPLE, feat, f"top1_sim<{TBM_DYNAMIC_MIN_TOP1_SIM:.2f}"

    budget = 1
    reasons = ["base=1"]
    length_dev = abs(feat["length_ratio"] - 1.0)

    if feat["top1_sim"] >= TBM_DYNAMIC_LEVEL2_SIM and feat["top1_pct_id"] >= TBM_DYNAMIC_LEVEL2_PCT:
        budget = max(budget, 2)
        reasons.append("level2")
    if (
        feat["top1_sim"] >= TBM_DYNAMIC_LEVEL3_SIM
        and feat["top1_pct_id"] >= TBM_DYNAMIC_LEVEL3_PCT
        and feat["alignment_gap_fraction"] <= TBM_DYNAMIC_GOOD_GAP
    ):
        budget = max(budget, 3)
        reasons.append("level3")
    if (
        feat["top1_sim"] >= TBM_DYNAMIC_LEVEL4_SIM
        and feat["top1_pct_id"] >= TBM_DYNAMIC_LEVEL4_PCT
        and feat["top1_top2_gap"] >= TBM_DYNAMIC_MIN_TOP12_GAP
        and length_dev <= TBM_DYNAMIC_GOOD_LEN_RATIO_DEV
    ):
        budget = max(budget, 4)
        reasons.append("level4")
    if (
        feat["top1_sim"] >= TBM_DYNAMIC_LEVEL5_SIM
        and feat["top1_pct_id"] >= TBM_DYNAMIC_LEVEL5_PCT
        and feat["alignment_gap_fraction"] <= TBM_DYNAMIC_STRICT_GAP
        and length_dev <= TBM_DYNAMIC_STRICT_LEN_RATIO_DEV
        and feat["top1_top2_gap"] >= TBM_DYNAMIC_MIN_TOP12_GAP
    ):
        budget = N_SAMPLE
        reasons.append("level5")

    if feat["top1_top2_gap"] <= TBM_DYNAMIC_LOW_TOP12_GAP and budget > TBM_DYNAMIC_LOW_TOP12_GAP_MAX_BUDGET:
        budget = TBM_DYNAMIC_LOW_TOP12_GAP_MAX_BUDGET
        reasons.append("cap:low_top12_gap")
    if feat["alignment_gap_fraction"] >= TBM_DYNAMIC_HIGH_GAP_THRESHOLD and budget > TBM_DYNAMIC_HIGH_GAP_MAX_BUDGET:
        budget = TBM_DYNAMIC_HIGH_GAP_MAX_BUDGET
        reasons.append("cap:high_gap")
    if length_dev >= TBM_DYNAMIC_BAD_LEN_RATIO_DEV and budget > TBM_DYNAMIC_BAD_LEN_RATIO_MAX_BUDGET:
        budget = TBM_DYNAMIC_BAD_LEN_RATIO_MAX_BUDGET
        reasons.append("cap:length_ratio")

    n_tbm_keep = int(np.clip(budget, 0, N_SAMPLE))
    n_protenix_request = int(np.clip(N_SAMPLE - n_tbm_keep, 0, N_SAMPLE))
    return n_tbm_keep, n_protenix_request, feat, "; ".join(reasons)


def decide_tbm_prediction_budget(seq_len, similar, features=None):
    n_tbm_keep, _, feat, reason = decide_tbm_vs_protenix_allocation(seq_len, similar, features)
    return n_tbm_keep, feat, reason


def _fill_missing_coords(new_coords: np.ndarray) -> np.ndarray:
    out = new_coords.copy()
    for i in range(len(out)):
        if np.isnan(out[i, 0]):
            pv = next((j for j in range(i - 1, -1, -1) if not np.isnan(out[j, 0])), -1)
            nv = next((j for j in range(i + 1, len(out)) if not np.isnan(out[j, 0])), -1)
            if pv >= 0 and nv >= 0:
                w = (i - pv) / (nv - pv)
                out[i] = (1 - w) * out[pv] + w * out[nv]
            elif pv >= 0:
                out[i] = out[pv] + [3, 0, 0]
            elif nv >= 0:
                out[i] = out[nv] + [3, 0, 0]
            else:
                out[i] = [i * 3, 0, 0]
    return np.nan_to_num(out)


def _project_template_to_query(query_seq, template_seq, template_coords, alignment, fill_missing: bool = True) -> np.ndarray:
    new_coords = np.full((len(query_seq), 3), np.nan, dtype=np.float64)
    for (qs, qe), (ts, te) in zip(*alignment.aligned):
        chunk = template_coords[ts:te]
        if len(chunk) == (qe - qs):
            new_coords[qs:qe] = chunk
    if fill_missing:
        return _fill_missing_coords(new_coords)
    return new_coords


def adapt_template_to_query(query_seq, template_seq, template_coords) -> np.ndarray:
    aln = next(iter(_aligner.align(query_seq, template_seq)))
    return _project_template_to_query(query_seq, template_seq, template_coords, aln, fill_missing=True)


def _template_seq_similarity(seq_a: str, seq_b: str) -> float:
    if not seq_a or not seq_b:
        return 0.0
    aln = next(iter(_aligner.align(seq_a, seq_b)))
    ident = _count_identical_pairs(seq_a, seq_b, aln)
    return float(ident / max(min(len(seq_a), len(seq_b)), 1))


def select_template_candidates(similar, budget):
    if budget <= 0 or not similar:
        return []
    if not USE_DIVERSITY_TEMPLATE_SELECTION or budget == 1:
        return similar[:budget]

    selected = [similar[0]]
    remaining = list(similar[1:])
    while remaining and len(selected) < budget:
        best_idx = 0
        best_score = -1e18
        for idx, cand in enumerate(remaining):
            redundancy = max(_template_seq_similarity(cand[1], chosen[1]) for chosen in selected)
            mmr_score = DIVERSITY_MMR_ALPHA * float(cand[2]) - DIVERSITY_MMR_BETA * redundancy
            if mmr_score > best_score:
                best_score = mmr_score
                best_idx = idx
        selected.append(remaining.pop(best_idx))
    return selected


def _segment_pct_identity(query_seq, template_seq, qs, qe, ts, te) -> float:
    seg_len = max(qe - qs, 1)
    identical = sum(1 for qp, tp in zip(range(qs, qe), range(ts, te)) if query_seq[qp] == template_seq[tp])
    return 100.0 * identical / seg_len


def _extract_hybrid_segments(query_seq, template_seq, local_aln, local_score):
    if local_aln is None or local_score < HYBRID_TBM_MIN_LOCAL_SCORE:
        return []
    segments = []
    for (qs, qe), (ts, te) in zip(*local_aln.aligned):
        seg_len = qe - qs
        if seg_len < HYBRID_TBM_MIN_SEG_LEN:
            continue
        pct_id = _segment_pct_identity(query_seq, template_seq, qs, qe, ts, te)
        if pct_id < HYBRID_TBM_MIN_SEG_PCT_ID:
            continue
        segments.append({
            "q_range": (qs, qe),
            "t_range": (ts, te),
            "seg_len": seg_len,
            "pct_id": pct_id,
            "local_score": local_score,
        })
    segments.sort(key=lambda x: (x["local_score"], x["seg_len"], x["pct_id"]), reverse=True)
    return segments


def _align_donor_fragment(base_coords, donor_coords, qs, qe):
    aligned = donor_coords.copy()
    left = max(0, qs - HYBRID_TBM_ANCHOR_PAD)
    right = min(len(base_coords), qe + HYBRID_TBM_ANCHOR_PAD)
    anchor_idx = [i for i in range(left, right) if (i < qs or i >= qe) and not np.isnan(donor_coords[i, 0]) and not np.isnan(base_coords[i, 0])]
    if len(anchor_idx) < 3:
        anchor_idx = [i for i in range(qs, qe) if not np.isnan(donor_coords[i, 0]) and not np.isnan(base_coords[i, 0])]
    if len(anchor_idx) >= 3:
        R, t = kabsch_align(donor_coords[anchor_idx], base_coords[anchor_idx])
        aligned = (donor_coords @ R.T) + t
    return aligned


def _blend_hybrid_boundaries(base_coords, donor_coords, qs, qe):
    merged = base_coords.copy()
    merged[qs:qe] = donor_coords[qs:qe]
    blend = max(HYBRID_TBM_BOUNDARY_BLEND, 0)
    for offset in range(blend):
        left_idx = qs + offset
        right_idx = qe - offset - 1
        frac = (offset + 1) / (blend + 1)
        if left_idx < qe:
            merged[left_idx] = (1 - frac) * base_coords[left_idx] + frac * donor_coords[left_idx]
        if right_idx >= qs:
            merged[right_idx] = (1 - frac) * base_coords[right_idx] + frac * donor_coords[right_idx]
    return merged


def apply_hybrid_tbm_lite(query_seq, base_template_id, base_coords, similar, used_template_ids=None):
    if not USE_HYBRID_TBM_LITE:
        return base_coords, []
    used_template_ids = set(used_template_ids or set())
    merged = base_coords.copy()
    logs = []
    covered = np.zeros(len(query_seq), dtype=bool)

    for donor_id, donor_seq, _, donor_coords, _, _, _, _, _, _ in similar[:HYBRID_TBM_MAX_DONORS + 1]:
        if donor_id == base_template_id or donor_id in used_template_ids:
            continue
        local_aln = next(iter(_local_aligner.align(query_seq, donor_seq)))
        local_score = _normalize_alignment_score(local_aln, len(query_seq), len(donor_seq)) if local_aln is not None else 0.0
        segments = _extract_hybrid_segments(query_seq, donor_seq, local_aln, local_score)
        if not segments:
            continue
        donor_partial = _project_template_to_query(query_seq, donor_seq, donor_coords, local_aln, fill_missing=False)
        donor_aligned = _align_donor_fragment(merged, donor_partial, segments[0]["q_range"][0], segments[0]["q_range"][1])
        for seg in segments:
            qs, qe = seg["q_range"]
            if covered[qs:qe].mean() > HYBRID_TBM_MAX_SEGMENT_COVERAGE:
                continue
            if np.isnan(donor_aligned[qs:qe]).any():
                continue
            merged = _blend_hybrid_boundaries(merged, donor_aligned, qs, qe)
            covered[qs:qe] = True
            logs.append({
                "donor_id": donor_id,
                "q_range": (qs, qe),
                "seg_len": seg["seg_len"],
                "pct_id": seg["pct_id"],
                "local_score": seg["local_score"],
            })
            if len(logs) >= HYBRID_TBM_MAX_FRAGMENTS:
                return merged, logs
    return merged, logs


# 得られた座標を，RNA らしい幾何に軽く寄せる後処理です．
# 隣接距離，i-i+2距離，平滑化，自己衝突回避を弱い拘束として複数回適用します．
def adaptive_rna_constraints(coords, target_id, segments_map, confidence=1.0, passes=2) -> np.ndarray:
    X        = coords.copy()
    segments = segments_map.get(target_id, [(0, len(X))])
    strength = max(0.75 * (1.0 - min(confidence, 0.97)), 0.02)
    for _ in range(passes):
        for s, e in segments:
            C = X[s:e]; L = e - s
            if L < 3:
                continue
            # bond i–i+1  ~5.95 Å
            d    = C[1:] - C[:-1]; dist = np.linalg.norm(d, axis=1) + 1e-6
            adj  = d * ((5.95 - dist) / dist)[:, None] * (0.22 * strength)
            C[:-1] -= adj; C[1:] += adj
            # soft i–i+2  ~10.2 Å
            d2   = C[2:] - C[:-2]; d2n = np.linalg.norm(d2, axis=1) + 1e-6
            adj2 = d2 * ((10.2 - d2n) / d2n)[:, None] * (0.10 * strength)
            C[:-2] -= adj2; C[2:] += adj2
            # Laplacian smoothing
            C[1:-1] += (0.06 * strength) * (0.5 * (C[:-2] + C[2:]) - C[1:-1])
            # self-avoidance
            if L >= 25:
                idx  = np.linspace(0, L - 1, min(L, 160)).astype(int) if L > 220 else np.arange(L)
                P    = C[idx]; diff = P[:, None, :] - P[None, :, :]
                dm   = np.linalg.norm(diff, axis=2) + 1e-6
                sep  = np.abs(idx[:, None] - idx[None, :])
                mask = (sep > 2) & (dm < 3.2)
                if np.any(mask):
                    vec = (diff * ((3.2 - dm) / dm)[:, :, None] * mask[:, :, None]).sum(axis=1)
                    C[idx] += (0.015 * strength) * vec
            X[s:e] = C
    return X


# 任意軸まわりの回転行列を作る低レベル関数です．
# hinge 変形や chain 全体の微回転に利用します．
def _rotmat(axis, ang):
    a = np.asarray(axis, float); a /= np.linalg.norm(a) + 1e-12
    x, y, z = a; c, s = np.cos(ang), np.sin(ang); CC = 1 - c
    return np.array([[c+x*x*CC, x*y*CC-z*s, x*z*CC+y*s],
                     [y*x*CC+z*s, c+y*y*CC, y*z*CC-x*s],
                     [z*x*CC-y*s, z*y*CC+x*s, c+z*z*CC]])


# 長い鎖の途中を支点にして，後半部分を少しだけ折り曲げる変形です．
# TBM サンプルに多様性を持たせる目的で使います．
def apply_hinge(coords, seg, rng, deg=22):
    s, e = seg; L = e - s
    if L < 30: return coords
    pivot = s + int(rng.integers(10, L - 10))
    R = _rotmat(rng.normal(size=3), np.deg2rad(float(rng.uniform(-deg, deg))))
    X = coords.copy(); p0 = X[pivot].copy()
    X[pivot+1:e] = (X[pivot+1:e] - p0) @ R.T + p0
    return X


# chain ごとに小さな回転と平行移動を加えて，多量体配置の揺らぎを作ります．
# 全体の重心は維持しつつ，相対配置のみ少し変えます．
def jitter_chains(coords, segs, rng, deg=12, trans=1.5):
    X = coords.copy(); gc_ = X.mean(0, keepdims=True)
    for s, e in segs:
        R     = _rotmat(rng.normal(size=3), np.deg2rad(float(rng.uniform(-deg, deg))))
        shift = rng.normal(size=3); shift = shift / (np.linalg.norm(shift) + 1e-12) * float(rng.uniform(0, trans))
        c     = X[s:e].mean(0, keepdims=True)
        X[s:e] = (X[s:e] - c) @ R.T + c + shift
    X -= X.mean(0, keepdims=True) - gc_
    return X


# chain 内に滑らかな微小変形を入れます．
# ランダムノイズよりも連続性を保ちやすく，現実的な揺らぎに近づける狙いです．
def smooth_wiggle(coords, segs, rng, amp=0.8):
    X = coords.copy()
    for s, e in segs:
        L = e - s
        if L < 20: continue
        ctrl = np.linspace(0, L - 1, 6); disp = rng.normal(0, amp, (6, 3)); t = np.arange(L)
        X[s:e] += np.vstack([np.interp(t, ctrl, disp[:, k]) for k in range(3)]).T
    return X


# どの手法でも座標が得られない場合の最終フォールバックです．
# 理想化したらせん状の点列を生成し，提出欠損を防ぎます．
def generate_rna_structure(sequence: str, seed=None) -> np.ndarray:
    """Idealized A-form RNA helix — last-resort de-novo fallback."""
    if seed is not None:
        np.random.seed(seed)
    n = len(sequence); coords = np.zeros((n, 3))
    for i in range(n):
        ang = i * 0.6
        coords[i] = [10.0 * np.cos(ang), 10.0 * np.sin(ang), i * 2.5]
    return coords


def make_prediction_record(coords, source: str, confidence: float = 0.0, metadata: dict | None = None) -> dict:
    return {
        "coords": np.asarray(coords, dtype=np.float64),
        "source": source,
        "confidence": float(confidence),
        "metadata": dict(metadata or {}),
    }


def get_prediction_coords(pred):
    return pred["coords"] if isinstance(pred, dict) else pred


def cheap_physics_metrics(coords: np.ndarray) -> dict:
    coords = np.asarray(coords, dtype=np.float64)
    adj_penalty = _adjacent_distance_error(coords)
    clash_penalty = _self_clash_ratio(coords)
    centroid = coords.mean(axis=0, keepdims=True)
    rg = float(np.sqrt(np.mean(np.sum((coords - centroid) ** 2, axis=1))))
    expected_rg = max(2.2 * np.sqrt(max(len(coords), 1)), 1.0)
    compact_penalty = float(abs(rg - expected_rg) / expected_rg)
    score = (
        PHYSICS_WEIGHT_ADJ * adj_penalty
        + PHYSICS_WEIGHT_CLASH * clash_penalty
        + PHYSICS_WEIGHT_COMPACT * compact_penalty
    )
    return {
        "score": float(score),
        "adj_penalty": float(adj_penalty),
        "clash_penalty": float(clash_penalty),
        "compact_penalty": float(compact_penalty),
    }


def validate_coords(coords, label: str = "coords"):
    if coords is None:
        return False, f"{label}:none"
    arr = np.asarray(coords)
    if arr.ndim == 3:
        for idx in range(arr.shape[0]):
            ok, reason = validate_coords(arr[idx], f"{label}[{idx}]")
            if not ok:
                return ok, reason
        return True, "ok"
    if not USE_COORD_VALIDATION:
        return True, "validation=off"
    if arr.ndim != 2 or arr.shape[1] != 3:
        return False, f"{label}:bad_shape={arr.shape}"
    if not np.isfinite(arr).all():
        return False, f"{label}:nan_or_inf"
    if len(arr) >= 2:
        adj = np.linalg.norm(arr[1:] - arr[:-1], axis=1)
        if np.nanmax(adj) > COORD_MAX_ADJ_DISTANCE:
            return False, f"{label}:adjacent_distance>{COORD_MAX_ADJ_DISTANCE:.1f}"
    box = arr.max(axis=0) - arr.min(axis=0)
    if float(np.max(box)) > COORD_MAX_BOX_SIZE:
        return False, f"{label}:box>{COORD_MAX_BOX_SIZE:.1f}"
    if len(arr) >= 3:
        rounded = np.round(arr, 3)
        dup_ratio = 1.0 - (np.unique(rounded, axis=0).shape[0] / len(arr))
        if dup_ratio > COORD_MAX_DUPLICATE_RATIO:
            return False, f"{label}:duplicate_ratio={dup_ratio:.3f}"
        clash_ratio = _self_clash_ratio(arr)
        if clash_ratio > COORD_MAX_SELF_CLASH_RATIO:
            return False, f"{label}:self_clash_ratio={clash_ratio:.3f}"
    return True, "ok"


def should_use_boltz2_aux(routing_features: dict) -> bool:
    return (
        routing_features.get("top1_sim", 0.0) < BOLTZ2_WEAK_TOP1_SIM
        or routing_features.get("top1_pct_id", 0.0) < BOLTZ2_WEAK_TOP1_PCT
    )


def run_boltz2_aux_phase(boltz2_queue: dict) -> dict:
    boltz2_preds = {}
    if not USE_BOLTZ2_AUX or not boltz2_queue:
        return boltz2_preds
    code_dir = Path(BOLTZ2_CODE_DIR)
    model_dir = Path(BOLTZ2_MODEL_DIR)
    if not code_dir.exists() or not model_dir.exists() or not BOLTZ2_CMD:
        print(
            f"\nPHASE 2.5 skipped (Boltz-2 assets or command missing). "
            f"queue={len(boltz2_queue)}, code_dir={code_dir.exists()}, model_dir={model_dir.exists()}, cmd={'set' if BOLTZ2_CMD else 'unset'}"
        )
        return boltz2_preds
    print(f"\nPHASE 2.5 placeholder: Boltz-2 queue prepared for {len(boltz2_queue)} target(s)")
    for tid, payload in boltz2_queue.items():
        print(f"  {tid}: Boltz-2 auxiliary branch configured but no local runner hook was exercised in this environment")
    return boltz2_preds


def rerank_candidates_with_physics(target_id: str, candidates: list) -> list:
    if not USE_PHYSICS_RERANK:
        return candidates
    scored = []
    for cand in candidates:
        coords = get_prediction_coords(cand)
        metrics = cheap_physics_metrics(coords)
        merged = dict(cand)
        merged["physics"] = metrics
        meta = dict(merged.get("metadata", {}))
        meta["physics_score"] = metrics["score"]
        merged["metadata"] = meta
        print(
            f"  {target_id}: candidate src={cand['source']}, conf={cand['confidence']:.3f}, "
            f"physics={metrics['score']:.3f} (adj={metrics['adj_penalty']:.3f}, clash={metrics['clash_penalty']:.4f}, compact={metrics['compact_penalty']:.3f})"
        )
        scored.append(merged)
    scored.sort(key=lambda x: (x["physics"]["score"], -x["confidence"]))
    return scored


def select_slot_ensemble(candidates: list, n_sample: int) -> list:
    if not candidates:
        return []
    if not USE_SLOT_ENSEMBLE:
        return candidates[:n_sample]
    primary_sources = {"tbm", "hybrid_tbm_lite", "protenix"}
    primary = [c for c in candidates if c["source"] in primary_sources]
    aux = [c for c in candidates if c["source"] not in primary_sources]
    primary.sort(key=lambda x: (-x["confidence"], x["metadata"].get("physics_score", 0.0)))
    aux.sort(key=lambda x: (-x["confidence"], x["metadata"].get("physics_score", 0.0)))
    selected = primary[:SLOT_ENSEMBLE_PRIMARY_SLOTS]
    seen = {id(c) for c in selected}
    for cand in primary[SLOT_ENSEMBLE_PRIMARY_SLOTS:] + aux:
        if len(selected) >= n_sample:
            break
        if id(cand) in seen:
            continue
        selected.append(cand)
        seen.add(id(cand))
    return selected[:n_sample]


def summarize_prediction_sources(candidates: list) -> str:
    if not candidates:
        return "none"
    return ", ".join(
        f"slot{i+1}={cand['source']}@{cand['confidence']:.3f}" for i, cand in enumerate(candidates)
    )


# Phase 1 では，既知構造を使った TBM を実行します．
# 各 target ごとに template 予測を作り，不足分だけ Protenix に回します．

# ─────────────── TBM Phase ───────────────────────────────────────────────────
# TBM を全 target に適用し，template だけで埋まるものと Protenix 補完が必要なものを振り分けます．
# 戻り値は，TBM 予測群と Protenix 実行待ちキューです．
def tbm_phase(test_df, train_seqs_df, train_coords_dict, segments_map):
    """
    Phase 1 — Template-Based Modeling.

    Returns
    -------
    template_predictions : {target_id: [prediction_record, ...]}
        0 to N_SAMPLE predictions per target, from real templates.
    protenix_queue : {target_id: (n_needed, full_sequence)}
        Targets that still need more predictions.
    boltz2_queue : {target_id: payload}
        Weak targets that may receive auxiliary Boltz-2 predictions.
    """
    print(f"\n{'='*60}")
    print(f"PHASE 1: Template-Based Modeling")
    print(f"  MIN_SIMILARITY = {MIN_SIMILARITY}")
    print(f"  USE_ADAPTIVE_IDENTITY_THRESHOLD = {USE_ADAPTIVE_IDENTITY_THRESHOLD}")
    print(f"  USE_DYNAMIC_ROUTING = {USE_DYNAMIC_ROUTING}")
    print(f"  USE_MULTI_RETRIEVAL = {USE_MULTI_RETRIEVAL}")
    print(f"  USE_HYBRID_TBM_LITE = {USE_HYBRID_TBM_LITE}")
    print(f"  USE_DIVERSITY_TEMPLATE_SELECTION = {USE_DIVERSITY_TEMPLATE_SELECTION}")
    print(f"  USE_EXTERNAL_TEMPLATES = {USE_EXTERNAL_TEMPLATES}")
    print(f"  Template library size = {len(train_coords_dict)}")
    print(f"{'='*60}")
    t0 = time.time()

    template_predictions: dict = {}
    protenix_queue: dict = {}
    boltz2_queue: dict = {}

    for _, row in test_df.iterrows():
        tid = row["target_id"]
        seq = row["sequence"]
        segs = segments_map.get(tid, [(0, len(seq))])

        similar = find_similar_sequences_detailed(seq, train_seqs_df, train_coords_dict, top_n=30)
        preds = []
        used = set()

        routing_features = summarize_tbm_routing_features(len(seq), similar)
        target_required_pct = required_pct_identity(len(seq), routing_features["alignment_gap_fraction"])
        target_tbm_budget, target_protenix_request, routing_features, routing_reason = decide_tbm_vs_protenix_allocation(
            len(seq), similar, routing_features
        )
        top_retrieval_summary = _format_retrieval_summary(similar[0]) if similar else "none"
        print(
            f"  {tid}: top1_sim={routing_features['top1_sim']:.3f}, "
            f"top2_sim={routing_features['top2_sim']:.3f}, "
            f"top1_pct_id={routing_features['top1_pct_id']:.1f}, "
            f"top1_top2_gap={routing_features['top1_top2_gap']:.3f}, "
            f"length_ratio={routing_features['length_ratio']:.3f}, "
            f"alignment_gap_fraction={routing_features['alignment_gap_fraction']:.3f}, "
            f"pct_threshold={target_required_pct:.1f}, retrieval={top_retrieval_summary} -> "
            f"TBM target {target_tbm_budget}, Protenix target {target_protenix_request}; "
            f"reason: {routing_reason}"
        )

        candidate_templates = select_template_candidates(similar, max(target_tbm_budget, HYBRID_TBM_MAX_DONORS + 1))
        for i, (tmpl_id, tmpl_seq, sim, tmpl_coords, pct_id, gap_fraction, length_ratio, _, _, retrieval_sources) in enumerate(candidate_templates):
            if len(preds) >= target_tbm_budget:
                break
            if sim < MIN_SIMILARITY:
                break
            required_pct = required_pct_identity(len(seq), gap_fraction)
            if pct_id < required_pct:
                continue
            if tmpl_id in used:
                continue

            rng = np.random.default_rng((row.name * 10000000000 + i * 10007) % (2**32))
            adapted = adapt_template_to_query(seq, tmpl_seq, tmpl_coords)
            hybrid_logs = []
            source_name = "tbm"
            if USE_HYBRID_TBM_LITE:
                adapted, hybrid_logs = apply_hybrid_tbm_lite(seq, tmpl_id, adapted, similar, used_template_ids=used)
                if hybrid_logs:
                    source_name = "hybrid_tbm_lite"
                    for h in hybrid_logs:
                        qs, qe = h["q_range"]
                        print(
                            f"    {tid}: hybrid range {qs}:{qe} <- {h['donor_id']} "
                            f"(len={h['seg_len']}, pct_id={h['pct_id']:.1f}, local={h['local_score']:.3f})"
                        )

            slot = len(preds)
            if slot == 0:
                X = adapted
            elif slot == 1:
                X = adapted + rng.normal(0, max(0.01, (0.40 - sim) * 0.06), adapted.shape)
            elif slot == 2:
                longest = max(segs, key=lambda se: se[1] - se[0])
                X = apply_hinge(adapted, longest, rng)
            elif slot == 3:
                X = jitter_chains(adapted, segs, rng)
            else:
                X = smooth_wiggle(adapted, segs, rng)

            refined = adaptive_rna_constraints(X, tid, segments_map, confidence=sim)
            ok, reason = validate_coords(refined, f"tbm:{tid}:{tmpl_id}")
            if not ok:
                print(f"    {tid}: rejected {tmpl_id} because {reason}")
                continue
            preds.append(make_prediction_record(
                refined,
                source=source_name,
                confidence=sim,
                metadata={
                    "template_id": tmpl_id,
                    "pct_id": pct_id,
                    "gap_fraction": gap_fraction,
                    "length_ratio": length_ratio,
                    "retrieval_sources": retrieval_sources,
                    "hybrid_logs": hybrid_logs,
                },
            ))
            used.add(tmpl_id)

        template_predictions[tid] = preds
        n_needed = N_SAMPLE - len(preds)
        if n_needed > 0:
            protenix_queue[tid] = (n_needed, seq)
            print(
                f"  {tid} ({len(seq)} nt): planned {target_tbm_budget} TBM, "
                f"accepted {len(preds)} TBM → need {n_needed} from Protenix"
            )
        else:
            print(f"  {tid} ({len(seq)} nt): planned {target_tbm_budget} TBM, all {N_SAMPLE} covered by TBM ✓")

        if should_use_boltz2_aux(routing_features):
            boltz2_queue[tid] = {
                "sequence": seq,
                "max_predictions": min(BOLTZ2_MAX_PRED_PER_TARGET, N_SAMPLE),
                "routing_features": routing_features,
            }
            print(f"  {tid}: queued for Boltz-2 auxiliary branch (weak target heuristic)")

    elapsed = time.time() - t0
    n_full = len(test_df) - len(protenix_queue)
    print(f"\nPhase 1 done in {elapsed:.1f}s")
    print(f"  Fully covered by TBM : {n_full}")
    print(f"  Need Protenix        : {len(protenix_queue)}")
    print(f"  Boltz-2 aux queue    : {len(boltz2_queue)}")
    return template_predictions, protenix_queue, boltz2_queue


# main() はこの notebook 全体の制御塔です．
# データ読込，TBM，Protenix，統合，submission 保存までを順に実行します．

# ─────────────── Main ────────────────────────────────────────────────────────
# 全体の実行フローをまとめたメイン関数です．
# Phase 1 で TBM，Phase 2 で Protenix，Phase 3 で統合と保存を行います．
def main() -> None:
    test_csv, output_csv, code_dir, root_dir = resolve_paths()

    if not os.path.isdir(code_dir):
        raise FileNotFoundError(
            f"Missing PROTENIX_CODE_DIR: {code_dir}. "
            "Set PROTENIX_CODE_DIR to the repo path."
        )

    os.environ["PROTENIX_ROOT_DIR"] = root_dir
    sys.path.append(code_dir)
    ensure_required_files(root_dir)
    seed_everything(SEED)

    # まず提出対象の test 配列を読み込みます．
    # ローカル実行時は件数制限をかけ，本番では全件を処理します．
    # ── Load test data ──────────────────────────────────────────────────────
    test_df_full = pd.read_csv(test_csv)
    test_df      = (test_df_full.head(LOCAL_N_SAMPLES) if not IS_KAGGLE
                    else test_df_full).reset_index(drop=True)
    print(f"Test targets : {len(test_df)}"
          + (" (LOCAL MODE)" if not IS_KAGGLE else ""))

    seq_by_id = dict(zip(test_df["target_id"], test_df["sequence"]))

    # Truncated copy for Protenix (Protenix has token limits)
    test_df_trunc = test_df.copy()
    test_df_trunc["sequence"] = test_df_trunc["sequence"].str[:MAX_SEQ_LEN]

    # TBM で参照する既知構造ライブラリを作るため，train / validation を結合して読み込みます．
    # 座標は target_id ごとの辞書へ整理し，template 探索をしやすくします．
    # ── Load training data for TBM ──────────────────────────────────────────
    print("\nLoading training data for TBM …")
    train_seqs   = pd.read_csv(DEFAULT_TRAIN_CSV)
    val_seqs     = pd.read_csv(DEFAULT_VAL_CSV)
    train_labels = pd.read_csv(DEFAULT_TRAIN_LBLS)
    val_labels   = pd.read_csv(DEFAULT_VAL_LBLS)

    combined_seqs   = pd.concat([train_seqs,   val_seqs],    ignore_index=True)
    combined_labels = pd.concat([train_labels, val_labels],  ignore_index=True)
    train_coords    = process_labels(combined_labels)
    combined_seqs, train_coords, _ = merge_external_templates_into_pool(combined_seqs, train_coords)
    segments_map, _ = build_segments_map(test_df)

    print(f"Template pool: {len(combined_seqs)} sequences, {len(train_coords)} structures")

    # Phase 1: 既知構造ベースの予測を先に作り，深層モデルに渡す件数を削減します．
    # ─── PHASE 1: TBM ──────────────────────────────────────────────────────
    template_preds, protenix_queue, boltz2_queue = tbm_phase(
        test_df, combined_seqs, train_coords, segments_map
    )

    # Phase 2: TBM だけで埋まらなかった target に対してのみ Protenix を走らせます．
    # 長鎖はチャンク分割し，後で全長へつなぎ直します．
    # ─── PHASE 2: Protenix (only for targets that need extra predictions) ──
    protenix_preds: dict = {}   # target_id -> np.ndarray (n_needed, seq_len, 3)
    boltz2_preds: dict = {}

    if protenix_queue and USE_PROTENIX:
        print(f"\n{'='*60}")
        print(f"PHASE 2: Protenix for {len(protenix_queue)} targets")
        print(f"{'='*60}")

        work_dir = Path(output_csv).resolve().parent
        work_dir.mkdir(parents=True, exist_ok=True)

        # 1段目では，Protenix に流す対象を単一配列またはチャンク単位の task に展開します．
        # 長鎖はそのままでは入らないため，あらかじめ分割してキュー化します．
        # ── 1. Preparation: create tasks for all sequences/chunks ────────────
        tasks = []          # list of dict for input_json
        chunk_info = {}     # target_id -> list of {"name": chunk_name, "range": (s, e)}
        
        for target_id, (n_needed, full_seq) in protenix_queue.items():
            seq_len = len(full_seq)
            if seq_len <= MAX_SEQ_LEN:
                tasks.append({"target_id": target_id, "sequence": full_seq})
                chunk_info[target_id] = [{"name": target_id, "range": (0, seq_len)}]
                print(f"  {target_id} ({seq_len} nt): single pass queued")
            else:
                chunks = split_into_chunks(seq_len, MAX_SEQ_LEN, CHUNK_OVERLAP)
                print(f"  {target_id} ({seq_len} nt): {len(chunks)} chunks queued "
                      f"{[(s, e) for s, e in chunks]}")
                
                chunk_info[target_id] = []
                for ci, (cs, ce) in enumerate(chunks):
                    chunk_name = f"{target_id}_chunk{ci}"
                    sub_seq = full_seq[cs:ce]
                    tasks.append({"target_id": chunk_name, "sequence": sub_seq})
                    chunk_info[target_id].append({"name": chunk_name, "range": (cs, ce)})

        # Build combined input JSON
        tasks_df = pd.DataFrame(tasks)
        input_json_path = str(work_dir / "protenix_queue_input.json")
        build_input_json(tasks_df, input_json_path)

        from protenix.data.inference.infer_dataloader import InferenceDataset
        from runner.inference import (InferenceRunner,
                                      update_gpu_compatible_configs,
                                      update_inference_configs)

        # Initialize model exactly ONCE
        configs = build_configs(input_json_path, str(work_dir / "outputs"), MODEL_NAME)
        configs = update_gpu_compatible_configs(configs)
        runner  = InferenceRunner(configs)
        dataset = InferenceDataset(configs)

        # 2段目では，task ごとに Protenix 推論を実行し，必要な C1' 座標だけを取り出します．
        # 推論失敗時は None を記録し，後でフォールバックできるようにします．
        # ── 2. Inference: process dataset and collect predictions ────────────
        raw_predictions = {}  # sample_name -> coords (np.ndarray or None)

        def _extract_c1_coords(prediction, feat, chunk_seq_len, raw_coords):
            if "centre_atom_mask" in feat:
                mask = (feat["centre_atom_mask"] == 1).to(raw_coords.device)
            elif "atom_to_tokatom_idx" in feat:
                m11 = (feat["atom_to_tokatom_idx"] == 11).to(raw_coords.device)
                m12 = (feat["atom_to_tokatom_idx"] == 12).to(raw_coords.device)
                c11, c12 = m11.sum(), m12.sum()
                mask = m11 if abs(c11 - chunk_seq_len) < abs(c12 - chunk_seq_len) else m12
            else:
                mask = torch.zeros(raw_coords.shape[1], dtype=torch.bool, device=raw_coords.device)
            
            coords = raw_coords[:, mask, :].detach().cpu().numpy()
            
            # Collapse check
            if coords.shape[1] > 1:
                diffs = np.linalg.norm(coords[0, 1:] - coords[0, :-1], axis=-1)
                if np.all(diffs < 1e-4):
                    print(f"    WARNING: Collapsed coordinates detected")
                    return None
            
            if coords.shape[1] != chunk_seq_len:
                if coords.shape[1] == 1 and chunk_seq_len > 1:
                    return None
                padded = np.zeros((coords.shape[0], chunk_seq_len, 3), dtype=np.float32)
                ml = min(coords.shape[1], chunk_seq_len)
                padded[:, :ml, :] = coords[:, :ml, :]
                coords = padded
            ok, reason = validate_coords(coords, f"protenix_extract:{chunk_seq_len}")
            if not ok:
                print(f"    WARNING: rejected extracted coords because {reason}")
                return None
            return coords

        for i in tqdm(range(len(dataset)), desc="Protenix Inference"):
            data, atom_array, err = dataset[i]
            sample_name = data.get("sample_name", f"sample_{i}")
            
            if err:
                print(f"  {sample_name} data error: {err}")
                raw_predictions[sample_name] = None
                del data, atom_array, err
                gc.collect(); torch.cuda.empty_cache(); gc.collect()
                continue
            
            # Find how many samples are needed for this specific query
            target_id = sample_name.split("_chunk")[0] if "_chunk" in sample_name else sample_name
            n_needed = protenix_queue.get(target_id, (N_SAMPLE, ""))[0]
            sub_seq_len = data["N_token"].item() # roughly correct
            
            try:
                new_cfg = update_inference_configs(configs, sub_seq_len)
                new_cfg.sample_diffusion.N_sample = n_needed
                runner.update_model_configs(new_cfg)
                
                pred = runner.predict(data)
                raw_coords = pred["coordinate"]
                
                coords = _extract_c1_coords(pred, data["input_feature_dict"], 
                                            sub_seq_len, raw_coords)
                raw_predictions[sample_name] = coords
            except Exception as exc:
                print(f"  {sample_name} inference failed: {exc}")
                import traceback; traceback.print_exc()
                raw_predictions[sample_name] = None
            finally:
                try: del pred, data, atom_array, raw_coords
                except: pass
                gc.collect(); torch.cuda.empty_cache(); gc.collect()

        # 3段目では，チャンク分割した長鎖RNAを全長座標へ復元します．
        # 短鎖はそのまま採用し，長鎖のみ stitch 処理を通します．
        # ── 3. Post-processing: Stitching and final formatting ───────────────
        for target_id, (n_needed, full_seq) in protenix_queue.items():
            seq_len = len(full_seq)
            chunks = chunk_info.get(target_id, [])
            
            if not chunks:
                continue

            if len(chunks) == 1:
                # Single pass
                coords = raw_predictions.get(target_id)
                ok, reason = validate_coords(coords, f"protenix:{target_id}")
                protenix_preds[target_id] = coords if ok else None
                if coords is not None and ok:
                    print(f"  {target_id}: {coords.shape[0]} predictions generated")
                else:
                    print(f"  {target_id}: FAILED ({reason})")
            else:
                # Stitch chunks together
                chunk_results_per_sample = {s: [] for s in range(n_needed)}
                all_ok = True
                
                for ci, cinfo in enumerate(chunks):
                    cname = cinfo["name"]
                    crange = cinfo["range"]
                    ccoords = raw_predictions.get(cname)
                    
                    if ccoords is None:
                        all_ok = False
                        break
                    
                    for s_idx in range(n_needed):
                        if s_idx < ccoords.shape[0]:
                            chunk_results_per_sample[s_idx].append((ccoords[s_idx], crange))
                        else:
                            chunk_results_per_sample[s_idx].append((ccoords[-1], crange))
                
                if not all_ok:
                    print(f"  {target_id}: chunked inference incomplete, using fallback")
                    protenix_preds[target_id] = None
                    continue
                
                stitched_samples = []
                for s_idx in range(n_needed):
                    items = chunk_results_per_sample[s_idx]
                    coords_list = [c for c, _ in items]
                    ranges_list = [r for _, r in items]
                    full_coords = stitch_chunk_coords(coords_list, ranges_list, seq_len)
                    ok, reason = validate_coords(full_coords, f"stitched:{target_id}:{s_idx}")
                    if ok:
                        stitched_samples.append(full_coords)
                    else:
                        print(f"  {target_id}: dropped stitched sample {s_idx} because {reason}")
                
                result = np.stack(stitched_samples, axis=0) if stitched_samples else None
                protenix_preds[target_id] = result
                if result is None:
                    print(f"  {target_id}: all stitched samples rejected")
                else:
                    print(f"  {target_id}: {result.shape[0]} stitched predictions generated")

    elif protenix_queue and not USE_PROTENIX:
        print(f"\nPHASE 2 skipped (USE_PROTENIX=False). "
              f"De-novo fallback will cover {len(protenix_queue)} targets.")

    # Phase 3: TBM，Protenix，de novo fallback を優先順に束ねて最終提出を作ります．
    # target ごとに必ず N_SAMPLE 本を揃えるのが目的です．
    # ─── PHASE 3: Combine everything ───────────────────────────────────────
    print(f"\n{'='*60}")
    print("PHASE 3: Combine TBM + Protenix + de-novo fallback")
    print(f"{'='*60}")

    all_rows = []

    for _, row in test_df.iterrows():
        tid = row["target_id"]
        seq = row["sequence"]

        candidates: list = list(template_preds.get(tid, []))

        ptx = protenix_preds.get(tid)
        if ptx is not None and ptx.ndim == 3:
            for j in range(ptx.shape[0]):
                candidates.append(make_prediction_record(
                    ptx[j],
                    source="protenix",
                    confidence=0.55 - 0.02 * j,
                    metadata={"sample_idx": j},
                ))

        btx = boltz2_preds.get(tid)
        if btx is not None and getattr(btx, "ndim", 0) == 3:
            for j in range(btx.shape[0]):
                candidates.append(make_prediction_record(
                    btx[j],
                    source="boltz2_aux",
                    confidence=0.35 - 0.01 * j,
                    metadata={"sample_idx": j},
                ))

        filtered = []
        for cand in candidates:
            ok, reason = validate_coords(get_prediction_coords(cand), f"final:{tid}:{cand['source']}")
            if ok:
                filtered.append(cand)
            else:
                print(f"  {tid}: dropped {cand['source']} because {reason}")
        candidates = rerank_candidates_with_physics(tid, filtered)
        combined = select_slot_ensemble(candidates, N_SAMPLE)

        n_denovo = 0
        while len(combined) < N_SAMPLE:
            seed_val = row.name * 1000000 + len(combined) * 1000
            dn = generate_rna_structure(seq, seed=seed_val)
            dn = adaptive_rna_constraints(dn, tid, segments_map, confidence=0.2)
            combined.append(make_prediction_record(
                dn,
                source="denovo",
                confidence=0.05,
                metadata={"seed": seed_val},
            ))
            n_denovo += 1

        if n_denovo:
            print(f"  {tid}: {n_denovo} slot(s) filled with de-novo fallback")
        print(f"  {tid}: final routing {summarize_prediction_sources(combined[:N_SAMPLE])}")

        stacked = np.stack([get_prediction_coords(pred) for pred in combined[:N_SAMPLE]], axis=0)
        all_rows.extend(coords_to_rows(tid, seq, stacked))

    # 最後に提出DataFrameを組み立て，列順と値域を整えて CSV として保存します．
    # ── Save ───────────────────────────────────────────────────────────────
    sub = pd.DataFrame(all_rows)
    cols = ["ID", "resname", "resid"] + [
        f"{c}_{i}" for i in range(1, N_SAMPLE + 1) for c in ["x", "y", "z"]
    ]
    coord_cols = [c for c in cols if c.startswith(("x_", "y_", "z_"))]
    sub[coord_cols] = sub[coord_cols].clip(-999.999, 9999.999)
    Path(output_csv).parent.mkdir(parents=True, exist_ok=True)
    sub[cols].to_csv(output_csv, index=False)

    print(f"\n✓ Saved submission to {output_csv}  ({len(sub):,} rows)")

/usr/local/lib/python3.12/dist-packages/Bio/Align/__init__.py:4414: BiopythonDeprecationWarning: The attribute 'query_left_open_gap_score' was renamed to 'open_left_deletion_score'. This was done to be consistent with the
AlignmentCounts object returned by the .counts method of an Alignment object.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/Bio/Align/__init__.py:4414: BiopythonDeprecationWarning: The attribute 'query_left_extend_gap_score' was renamed to 'extend_left_deletion_score'. This was done to be consistent with the
AlignmentCounts object returned by the .counts method of an Alignment object.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/Bio/Align/__init__.py:4414: BiopythonDeprecationWarning: The attribute 'query_right_open_gap_score' was renamed to 'open_right_deletion_score'. This was done to be consistent with the
AlignmentCounts object returned by the .counts method of an Alignment object.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/Bio

# Main

In [6]:
# main() を直接呼び出し，前セルまでに定義したパイプライン全体を実行します．


if __name__ == "__main__":
    main()

Test targets : 28

Loading training data for TBM …


/tmp/ipykernel_41/2371676172.py:646: DtypeWarning: Columns (6) have mixed types. Specify dtype option on import or set low_memory=False.
  train_labels = pd.read_csv(DEFAULT_TRAIN_LBLS)


Template pool: 5744 sequences, 5744 structures

PHASE 1: Template-Based Modeling
  MIN_SIMILARITY = 0.0  |  MIN_PCT_IDENTITY = 50.0
  8ZNQ (30 nt): 2 TBM → need 8 from Protenix
  9IWF (69 nt): all 10 from TBM ✓
  9JGM (210 nt): 5 TBM → need 5 from Protenix
  9MME (4640 nt): 4 TBM → need 6 from Protenix
  9J09 (214 nt): 1 TBM → need 9 from Protenix
  9E9Q (101 nt): 4 TBM → need 6 from Protenix
  9CFN (59 nt): 2 TBM → need 8 from Protenix
  9OBM (73 nt): 3 TBM → need 7 from Protenix
  9G4P (68 nt): 4 TBM → need 6 from Protenix
  9G4Q (104 nt): 2 TBM → need 8 from Protenix
  9G4R (47 nt): 1 TBM → need 9 from Protenix
  9RVP (34 nt): 4 TBM → need 6 from Protenix
  9JFS (246 nt): 1 TBM → need 9 from Protenix
  9LEC (378 nt): 2 TBM → need 8 from Protenix
  9LEL (476 nt): 2 TBM → need 8 from Protenix
  9I9W (28 nt): all 10 from TBM ✓
  9HRO (35 nt): all 10 from TBM ✓
  9QZJ (19 nt): all 10 from TBM ✓
  9JFO (195 nt): 5 TBM → need 5 from Protenix
  9OD4 (23 nt): 6 TBM → need 4 from Protenix
  

2026-03-12 10:29:49,080 [/kaggle/input/datasets/qiweiyin/protenix-v1-adjusted/Protenix-v1-adjust-v2/Protenix-v1-adjust-v2/Protenix-v1/runner/inference.py:246] INFO runner.inference: Distributed environment: world size: 1, global rank: 0, local rank: 0
2026-03-12 10:29:49,081 [/kaggle/input/datasets/qiweiyin/protenix-v1-adjusted/Protenix-v1-adjust-v2/Protenix-v1-adjust-v2/Protenix-v1/runner/inference.py:98] INFO root: LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]
2026-03-12 10:29:49,205 [/kaggle/input/datasets/qiweiyin/protenix-v1-adjusted/Protenix-v1-adjust-v2/Protenix-v1-adjust-v2/Protenix-v1/runner/inference.py:127] INFO root: Finished environment initialization.


train scheduler 16.0
inference scheduler 16.0
Diffusion Module has 16.0


2026-03-12 10:31:11,529 [/kaggle/input/datasets/qiweiyin/protenix-v1-adjusted/Protenix-v1-adjust-v2/Protenix-v1-adjust-v2/Protenix-v1/runner/inference.py:246] INFO runner.inference: Loading from /kaggle/input/datasets/qiweiyin/protenix-v1-adjusted/Protenix-v1-adjust-v2/Protenix-v1-adjust-v2/Protenix-v1/checkpoint/protenix_base_20250630_v1.0.0.pt, strict: True
2026-03-12 10:31:20,110 [/kaggle/input/datasets/qiweiyin/protenix-v1-adjusted/Protenix-v1-adjust-v2/Protenix-v1-adjust-v2/Protenix-v1/runner/inference.py:246] INFO runner.inference: Sampled key: module.input_embedder.atom_attention_encoder.linear_no_bias_ref_pos.weight
2026-03-12 10:31:20,253 [/kaggle/input/datasets/qiweiyin/protenix-v1-adjusted/Protenix-v1-adjust-v2/Protenix-v1-adjust-v2/Protenix-v1/runner/inference.py:246] INFO runner.inference: Finish loading checkpoint.
2026-03-12 10:31:20,265 [/kaggle/input/datasets/qiweiyin/protenix-v1-adjusted/Protenix-v1-adjust-v2/Protenix-v1-adjust-v2/Protenix-v1/runner/inference.py:246] 

  8ZNQ: 8 predictions generated
  9JGM: 5 predictions generated
  9MME: 6 stitched predictions generated
  9J09: 9 predictions generated
  9E9Q: 6 predictions generated
  9CFN: 8 predictions generated
  9OBM: 7 predictions generated
  9G4P: 6 predictions generated
  9G4Q: 8 predictions generated
  9G4R: 9 predictions generated
  9RVP: 6 predictions generated
  9JFS: 9 predictions generated
  9LEC: 8 predictions generated
  9LEL: 8 predictions generated
  9JFO: 5 predictions generated
  9OD4: 4 predictions generated
  9E74: 4 predictions generated
  9KGG: 4 predictions generated
  9EBP: 9 predictions generated
  9ZCC: 8 stitched predictions generated

PHASE 3: Combine TBM + Protenix + de-novo fallback

✓ Saved submission to /kaggle/working/submission.csv  (9,762 rows)


In [7]:
# 生成された submission.csv を読み戻し，先頭行を確認して形式が崩れていないかを目視チェックします．

submission_path = os.environ.get('SUBMISSION_CSV', DEFAULT_OUTPUT)
submission_df = pd.read_csv(submission_path)
print(submission_df.head(20))


         ID resname  resid        x_1        y_1        z_1         x_2  \
0    8ZNQ_1       A      1  -2.054246 -15.061790  20.740161  144.905975   
1    8ZNQ_2       C      2  -1.971328 -15.075652  15.338831  141.453103   
2    8ZNQ_3       C      3  -3.348014 -13.500002  10.449140  138.508506   
3    8ZNQ_4       G      4  -5.444428 -11.039888   6.602210  142.583253   
4    8ZNQ_5       U      5  -6.349629  -6.257964   4.544753  143.602598   
5    8ZNQ_6       G      6  -6.918016  -0.888219   3.240932  142.421563   
6    8ZNQ_7       A      7  -4.490872   4.125901  -0.914170  139.531180   
7    8ZNQ_8       C      8  -5.403762   9.778472   3.356724  137.092470   
8    8ZNQ_9       G      9  -0.394198  10.589010   0.112758  135.599965   
9   8ZNQ_10       G     10   3.902270   9.966390  -2.052086  134.618762   
10  8ZNQ_11       G     11   7.968209   7.519009  -4.491783  133.472268   
11  8ZNQ_12       C     12  10.186732   4.490251  -9.001780  132.259513   
12  8ZNQ_13       C     1